In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# ============================================================
# Single-cell: HEO + Rice (Cammeo & Osmancik) + LR tuning
# ============================================================

import os
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.exceptions import ConvergenceWarning

# ------------------------------------------------------------
# 1. Benchmark functions (for quick Sphere test)
# ------------------------------------------------------------

def sphere(x):
    return np.sum(x**2)

def step(x):
    return np.sum(np.floor(x + 0.5)**2)

def schwefel_221(x):
    return np.max(np.abs(x))

def schwefel_222(x):
    absx = np.abs(x)
    return np.sum(absx) + np.prod(absx)

def rosenbrock(x):
    return np.sum(100.0 * (x[1:] - x[:-1]**2)**2 + (1 - x[:-1])**2)

def bent_cigar(x):
    return x[0]**2 + 1e6*np.sum(x[1:]**2)

def sumsquares2(x):
    i = np.arange(1, x.size+1)
    return np.sum(i * x**2)

def alpine(x):
    return np.sum(np.abs(x * np.sin(x) + 0.1 * x))

def griewank(x):
    i = np.arange(1, x.size+1)
    return np.sum(x**2)/4000.0 - np.prod(np.cos(x/np.sqrt(i))) + 1.0

def rastrigin(x):
    return 10.0*x.size + np.sum(x**2 - 10.0*np.cos(2*np.pi*x))

def ackley(x):
    d = x.size
    a = 20.0
    b = 0.2
    c = 2*np.pi
    s1 = np.sum(x**2)
    s2 = np.sum(np.cos(c*x))
    term1 = -a * np.exp(-b*np.sqrt(s1/d))
    term2 = -np.exp(s2/d)
    return term1 + term2 + a + np.e

def levy(x):
    w = 1 + (x - 1)/4
    term1 = np.sin(np.pi*w[0])**2
    term3 = (w[-1] - 1)**2 * (1 + np.sin(2*np.pi*w[-1])**2)
    term2 = np.sum((w[:-1]-1)**2 * (1 + 10*np.sin(np.pi*w[:-1] + 1)**2))
    return term1 + term2 + term3

def salomon(x):
    r = np.sqrt(np.sum(x**2))
    return 1 - np.cos(2*np.pi*r) + 0.1*r

def schaffer(x):
    total = 0.0
    for i in range(x.size - 1):
        xi = x[i]
        xj = x[i+1]
        num = np.sin(np.sqrt(xi**2 + xj**2))**2 - 0.5
        den = (1 + 0.001*(xi**2 + xj**2))**2
        total += 0.5 + num/den
    return total

# ------------------------------------------------------------
# 2. HEO-style optimizer
# ------------------------------------------------------------

class HEO:
    """
    Halfway-Escape-Optimization-style optimizer (approximation).
    Minimizes f(x) over x in [lower, upper]^dim.
    """
    def __init__(
        self,
        func,
        dim,
        lower=-100.0,
        upper=100.0,
        n_particles=100,
        max_iters=1000,
        seed=None,
        escape_init=0.0,
        escape_increment=0.1,
        escape_max=2.0,
        stagnation_patience_global=30,
        stagnation_patience_vibration=10,
        skip_patience=80,
        w_global=0.5,
        sigma_radius=0.1,
        center_clip_factor=1.0,
        skip_scale=0.5,
    ):
        self.func = func
        self.dim = dim
        if np.isscalar(lower):
            self.lower = np.full(dim, float(lower))
        else:
            self.lower = np.array(lower, dtype=float)
        if np.isscalar(upper):
            self.upper = np.full(dim, float(upper))
        else:
            self.upper = np.array(upper, dtype=float)

        self.n_particles = n_particles
        self.max_iters = max_iters

        self.escape_factor = escape_init
        self.escape_increment = escape_increment
        self.escape_max = escape_max
        self.stagnation_patience_global = stagnation_patience_global
        self.stagnation_patience_vibration = stagnation_patience_vibration
        self.skip_patience = skip_patience
        self.w_global = w_global
        self.sigma_radius = sigma_radius
        self.center_clip_factor = center_clip_factor
        self.skip_scale = skip_scale

        self.rng = np.random.default_rng(seed)

        self.positions = None
        self.pbest = None
        self.pbest_f = None
        self.gbest = None
        self.gbest_f = None
        self.no_improve_local = None
        self.no_improve_global = 0
        self.energy = None
        self.it = 0

    def _init_swarm(self):
        self.positions = self.rng.uniform(
            self.lower, self.upper, size=(self.n_particles, self.dim)
        )
        self.pbest = self.positions.copy()
        self.pbest_f = np.apply_along_axis(self.func, 1, self.pbest)
        best_idx = np.argmin(self.pbest_f)
        self.gbest = self.pbest[best_idx].copy()
        self.gbest_f = self.pbest_f[best_idx]

        self.no_improve_local = np.zeros(self.n_particles, dtype=int)
        self.no_improve_global = 0
        self.energy = np.ones(self.n_particles, dtype=float)
        self.escape_factor = float(self.escape_factor)
        self.it = 0

    def _center_clipping(self, X):
        diffs = np.abs(X - self.gbest)
        spread = np.max(diffs, axis=0)
        r_soft = self.rng.random(self.dim)
        radius = self.center_clip_factor * (spread + 1e-12) * (0.5 + r_soft)
        domain_half = 0.5 * (self.upper - self.lower)
        radius = np.minimum(radius, domain_half)
        group_lower = np.maximum(self.gbest - radius, self.lower)
        group_upper = np.minimum(self.gbest + radius, self.upper)
        return np.clip(X, group_lower, group_upper)

    def _vibration(self, X):
        stagnant_mask = self.no_improve_local >= self.stagnation_patience_vibration
        if not np.any(stagnant_mask):
            return X

        std_vec = np.std(X, axis=0)
        std_vec = np.where(std_vec == 0, 1e-12, std_vec)

        for i in np.where(stagnant_mask)[0]:
            scale = self.sigma_radius * std_vec / (1.0 + self.energy[i])
            noise = self.rng.normal(loc=0.0, scale=scale, size=self.dim)
            X[i] = X[i] + noise
            self.energy[i] *= 1.02

        return X

    def _position_update(self, X):
        new_X = np.empty_like(X)
        for i in range(self.n_particles):
            x = X[i]
            p = self.pbest[i]
            g = self.gbest
            halfway = 0.5 * (p + g)
            dist_p = p - x
            dist_g = g - x
            r_escape = self.rng.exponential(1.0, size=self.dim)
            r_rand = self.rng.exponential(1.0, size=self.dim)

            if self.escape_factor <= 1e-12:
                step = self.w_global * (self.rng.random(self.dim) - 0.5) * (
                    np.abs(dist_p) + np.abs(dist_g)
                )
                x_new = halfway + step
            else:
                dir_vec = x - halfway
                if np.allclose(dir_vec, 0.0):
                    dir_vec = self.rng.normal(size=self.dim)
                step_escape = self.escape_factor * r_escape * dir_vec
                step_noise = self.w_global * r_rand * (dist_p + dist_g)
                x_new = x + step_escape + step_noise

            new_X[i] = x_new
        return new_X

    def _random_skip(self, X):
        if self.no_improve_global < self.skip_patience:
            return X

        domain_range = (self.upper - self.lower)
        radius = self.skip_scale * domain_range
        shift = self.rng.uniform(-radius, radius)
        center = np.clip(self.gbest + shift, self.lower, self.upper)
        low = np.maximum(center - radius, self.lower)
        high = np.minimum(center + radius, self.upper)
        X = self.rng.uniform(low, high, size=(self.n_particles, self.dim))

        self.no_improve_global = 0
        self.no_improve_local[:] = 0
        self.energy[:] = 1.0
        return X

    def run(self, verbose=False):
        self._init_swarm()
        for it in range(self.max_iters):
            self.it = it
            X = self._position_update(self.positions)
            X = self._vibration(X)
            X = self._center_clipping(X)
            X = self._random_skip(X)

            f_vals = np.apply_along_axis(self.func, 1, X)

            improved_local = f_vals < self.pbest_f
            self.pbest[improved_local] = X[improved_local]
            self.pbest_f[improved_local] = f_vals[improved_local]
            self.no_improve_local[improved_local] = 0
            self.no_improve_local[~improved_local] += 1

            best_idx = np.argmin(f_vals)
            best_f = f_vals[best_idx]
            if best_f + 1e-12 < self.gbest_f:
                self.gbest_f = best_f
                self.gbest = X[best_idx].copy()
                self.no_improve_global = 0
            else:
                self.no_improve_global += 1

            if self.no_improve_global >= self.stagnation_patience_global:
                self.escape_factor = min(
                    self.escape_factor + self.escape_increment, self.escape_max
                )
            else:
                self.escape_factor *= 0.99

            self.positions = X

            if verbose and (it % max(1, self.max_iters // 10) == 0):
                print(
                    f"iter={it:4d}, gbest_f={self.gbest_f:.4e}, escape={self.escape_factor:.3f}"
                )

        return self.gbest, self.gbest_f

# ------------------------------------------------------------
# 3. Load Rice dataset (your file)
# ------------------------------------------------------------

csv_path = "/content/Rice_data_type(1).csv"
df_rice = pd.read_csv(csv_path)
print("Using dataset:", csv_path)
print("Dataset shape:", df_rice.shape)
print(df_rice.head())

# last column is class label
target_col = df_rice.columns[-1]

# drop index-like columns explicitly
cols_to_drop = ["Unnamed: 0", "id", "ID"]
feature_cols = [c for c in df_rice.columns if c not in cols_to_drop + [target_col]]

X = df_rice[feature_cols].values.astype(float)
y_raw = df_rice[target_col].values

le = LabelEncoder()
y = le.fit_transform(y_raw)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ------------------------------------------------------------
# 4. LR eval + HEO optimization
# ------------------------------------------------------------

def lr_eval_from_vector(vec, X_tr, y_tr, X_val, y_val):
    """
    vec[0]: log10(C) in [-3, 3]
    vec[1]: scaled max_iter in [0, 1] -> [100, 2000]
    Returns loss = 1 - F1 (to minimize).
    """
    log10_C = vec[0]
    max_iter = int(100 + vec[1] * (2000 - 100))
    max_iter = max(100, min(max_iter, 2000))
    C = 10 ** log10_C

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=ConvergenceWarning)
        model = LogisticRegression(
            C=C,
            max_iter=max_iter,
            penalty="l2",
            solver="lbfgs",
        )
        model.fit(X_tr, y_tr)

    y_pred = model.predict(X_val)
    f1 = f1_score(y_val, y_pred)
    return 1.0 - f1

def heo_optimize_lr(
    X_train_scaled,
    y_train,
    n_particles=50,
    max_iters=50,
    seed=123,
):
    # inner validation split
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_scaled, y_train, test_size=0.2, random_state=42, stratify=y_train
    )

    lower = np.array([-3.0, 0.0])  # log10(C), scaled_iter
    upper = np.array([3.0, 1.0])

    def objective(v):
        return lr_eval_from_vector(v, X_tr, y_tr, X_val, y_val)

    heo = HEO(
        func=objective,
        dim=2,
        lower=lower,
        upper=upper,
        n_particles=n_particles,
        max_iters=max_iters,
        seed=seed,
        stagnation_patience_global=10,
        stagnation_patience_vibration=5,
        skip_patience=25,
        w_global=0.5,
        sigma_radius=0.1,
        center_clip_factor=1.0,
        skip_scale=0.5,
    )

    best_vec, best_loss = heo.run(verbose=True)

    # decode best hyperparameters consistently with lr_eval_from_vector
    log10_C = best_vec[0]
    max_iter = int(100 + best_vec[1] * (2000 - 100))
    max_iter = max(100, min(max_iter, 2000))
    C = 10 ** log10_C

    best_model = LogisticRegression(
        C=C,
        max_iter=max_iter,
        penalty="l2",
        solver="lbfgs",
    )
    best_model.fit(X_train_scaled, y_train)

    return best_model, best_vec, best_loss

# ------------------------------------------------------------
# 5. Run HEO on Rice dataset
# ------------------------------------------------------------

heo_model, best_vec, best_loss = heo_optimize_lr(
    X_train_scaled, y_train,
    n_particles=50,
    max_iters=50,
    seed=123,
)

print("\n=== HEO-tuned Logistic Regression ===")
print("Best vector (log10(C), scaled_iter):", best_vec)
print("Best validation loss (1 - F1):", best_loss)

y_pred_test = heo_model.predict(X_test_scaled)
acc = accuracy_score(y_test, y_pred_test)
f1 = f1_score(y_test, y_pred_test)

print(f"Test Accuracy: {acc:.4f}")
print(f"Test F1-score: {f1:.4f}")

# ------------------------------------------------------------
# 6. Quick Sphere test
# ------------------------------------------------------------

def quick_sphere_test():
    heo = HEO(
        func=sphere,
        dim=30,
        lower=-100.0,
        upper=100.0,
        n_particles=30,
        max_iters=200,
        seed=1,
    )
    _, best_f = heo.run(verbose=False)
    print("\nQuick Sphere test (30D) best_f:", best_f)

quick_sphere_test()


Using dataset: /content/Rice_data_type(1).csv
Dataset shape: (3810, 9)
   Unnamed: 0     Area   Perimeter  Major_Axis_Length  Minor_Axis_Length  \
0           0  15231.0  525.578979         229.749878          85.093788   
1           1  14656.0  494.311005         206.020065          91.730972   
2           2  14634.0  501.122009         214.106781          87.768288   
3           3  13176.0  458.342987         193.337387          87.448395   
4           4  14688.0  507.166992         211.743378          89.312454   

   Eccentricity  Convex_Area    Extent      Class  
0      0.928882      15617.0  0.572896  b'Cammeo'  
1      0.895405      15072.0  0.615436  b'Cammeo'  
2      0.912118      14954.0  0.693259  b'Cammeo'  
3      0.891861      13368.0  0.640669  b'Cammeo'  
4      0.906691      15262.0  0.646024  b'Cammeo'  
iter=   0, gbest_f=4.5317e-02, escape=0.000
iter=   5, gbest_f=4.5317e-02, escape=0.000
iter=  10, gbest_f=4.5317e-02, escape=0.100
iter=  15, gbest_f=4.5317e-0

KeyboardInterrupt: 

In [ ]:
import numpy as np
import pandas as pd
import time

# ============================================================
# 1. Benchmark functions (matching Tables 1 & 2, 30D, [-100,100])
# ============================================================

def sphere(x):
    return np.sum(x**2)

def step(x):
    return np.sum(np.floor(x + 0.5)**2)

def schwefel_221(x):
    return np.max(np.abs(x))

def schwefel_222(x):
    absx = np.abs(x)
    return np.sum(absx) + np.prod(absx)

def rosenbrock(x):
    return np.sum(100.0 * (x[1:] - x[:-1]**2)**2 + (1 - x[:-1])**2)

def bent_cigar(x):
    return x[0]**2 + 1e6*np.sum(x[1:]**2)

def sumsquares2(x):
    i = np.arange(1, x.size+1)
    return np.sum(i * x**2)

def alpine(x):
    return np.sum(np.abs(x * np.sin(x) + 0.1 * x))

def griewank(x):
    i = np.arange(1, x.size+1)
    return np.sum(x**2)/4000.0 - np.prod(np.cos(x/np.sqrt(i))) + 1.0

def rastrigin(x):
    return 10.0*x.size + np.sum(x**2 - 10.0*np.cos(2*np.pi*x))

def ackley(x):
    d = x.size
    a = 20.0
    b = 0.2
    c = 2*np.pi
    s1 = np.sum(x**2)
    s2 = np.sum(np.cos(c*x))
    term1 = -a * np.exp(-b*np.sqrt(s1/d))
    term2 = -np.exp(s2/d)
    return term1 + term2 + a + np.e

def levy(x):
    w = 1 + (x - 1)/4
    term1 = np.sin(np.pi*w[0])**2
    term3 = (w[-1] - 1)**2 * (1 + np.sin(2*np.pi*w[-1])**2)
    term2 = np.sum((w[:-1]-1)**2 * (1 + 10*np.sin(np.pi*w[:-1] + 1)**2))
    return term1 + term2 + term3

def salomon(x):
    r = np.sqrt(np.sum(x**2))
    return 1 - np.cos(2*np.pi*r) + 0.1*r

def schaffer(x):
    total = 0.0
    for i in range(x.size - 1):
        xi = x[i]
        xj = x[i+1]
        num = np.sin(np.sqrt(xi**2 + xj**2))**2 - 0.5
        den = (1 + 0.001*(xi**2 + xj**2))**2
        total += 0.5 + num/den
    return total

benchmark_funcs = [
    ("Sphere", sphere),
    ("Step", step),
    ("Schwefel 2.21", schwefel_221),
    ("Schwefel 2.22", schwefel_222),
    ("Rosenbrock", rosenbrock),
    ("BentCigar", bent_cigar),
    ("Sumsquares2", sumsquares2),
    ("Alpine", alpine),
    ("Griewank", griewank),
    ("Rastrigin", rastrigin),
    ("Ackley", ackley),
    ("Levy", levy),
    ("Salomon", salomon),
    ("Schaffer", schaffer),
]

# ============================================================
# 2. Paper-style HEO implementation (Algorithm 1 + eqs)
# ============================================================

class HEO:
    """
    Halfway Escape Optimization (HEO) — implementation close to Algorithm 1
    and equations (1–7), (13–22) in the paper.

    - Position update: eqs (1–3) with r1,r2,r3
    - r1 ~ U(1-R, 1+R), r2 ~ U(0.5,1.5), r3 ~ U(0,1)
    - Global counter c (escape factor) used in position update
    - Per-quantum energy a_i for Vibration (eqs (13–16))
    - Center clipping (eqs (17–19)) using L_inf norm
    - Random Skip (eqs (20–22)) to explore new region
    """

    def __init__(
        self,
        func,
        dim,
        lower=-100.0,
        upper=100.0,
        swarm_size=100,
        max_iters=1000,
        R=1.0,          # controls r1 range: [1-R, 1+R]
        a_max=10.0,     # upper scale for energy a_i
        c_max=30,       # when c > c_max -> random skip
        seed=None,
    ):
        self.func = func
        self.dim = dim
        self.swarm_size = swarm_size
        self.max_iters = max_iters
        self.R = R
        self.a_max = a_max
        self.c_max = c_max

        if np.isscalar(lower):
            self.lower = np.full(dim, float(lower))
        else:
            self.lower = np.array(lower, dtype=float)

        if np.isscalar(upper):
            self.upper = np.full(dim, float(upper))
        else:
            self.upper = np.array(upper, dtype=float)

        self.rng = np.random.default_rng(seed)

        # internal state
        self.positions = None
        self.local_best_pos = None
        self.local_best_cost = None
        self.global_best_pos = None
        self.global_best_cost = None

        self.energy = None  # a_i
        self.c_global = 0   # c_i

    def _init_swarm(self):
        self.positions = self.rng.uniform(
            self.lower, self.upper, size=(self.swarm_size, self.dim)
        )
        # initial personal bests
        self.local_best_pos = self.positions.copy()
        self.local_best_cost = np.apply_along_axis(self.func, 1, self.local_best_pos)

        # initial global best
        idx = np.argmin(self.local_best_cost)
        self.global_best_pos = self.local_best_pos[idx].copy()
        self.global_best_cost = self.local_best_cost[idx]

        # energy levels for vibration
        self.energy = np.zeros(self.swarm_size, dtype=float)

        # global escape counter
        self.c_global = 0

    def _position_update(self, X):
        """
        Position update (eqs (1–3)).
        x_{i+1} = x_i + v_g + v_l
        v_g = (x_g - x_i (c+1) * r1) * r2 * r3
        v_l = (x_l - x_i (c+1) * r1) * r2 * (1 - r3)
        """
        k, d = X.shape

        r1 = self.rng.uniform(1.0 - self.R, 1.0 + self.R, size=(k, d))
        r2 = self.rng.uniform(0.5, 1.5, size=(k, d))
        r3 = self.rng.uniform(0.0, 1.0, size=(k, d))

        c_factor = (self.c_global + 1.0)

        xg = self.global_best_pos
        xl = self.local_best_pos

        # broadcasting
        term = X * (c_factor * r1)
        v_g = (xg - term) * r2 * r3
        v_l = (xl - term) * r2 * (1.0 - r3)

        return X + v_g + v_l

    def _vibration(self, X, vib_mask):
        """
        Vibration (eq (13)):
            x_{i+1} = x_i + n / (1 + e^{a_i})
        where n ~ N(0, σ_x), σ_x approximated as std of the swarm.
        """
        if not np.any(vib_mask):
            return X

        # standard deviation across swarm for each dimension
        std_vec = np.std(X, axis=0)
        std_vec = np.where(std_vec == 0.0, 1e-12, std_vec)

        idxs = np.where(vib_mask)[0]
        for idx in idxs:
            # eq16: n ~ N(0, σ_x_i)
            n = self.rng.normal(loc=0.0, scale=std_vec, size=self.dim)
            scale = 1.0 / (1.0 + np.exp(self.energy[idx]))  # 1/(1+e^{a_i})
            X[idx] = X[idx] + scale * n

        return X

    def _center_clipping(self, X):
        """
        Center clipping (eqs (17–19)):
            S_result = S_bound ∩ S_group
            b_y = ||x_i - x_g|| * r5,   r5 ~ U(0,2)
        We use L_inf norm to define b_y as a scalar radius, then clip to a box
        around x_g with half-side b_y.
        """
        k, d = X.shape
        X_clipped = np.empty_like(X)
        for i in range(k):
            xi = X[i]
            # distance from global best
            diff = xi - self.global_best_pos
            # L_inf norm
            dist = np.max(np.abs(diff))
            r5 = self.rng.uniform(0.0, 2.0)
            b_y = dist * r5

            # build group box
            lower_group = self.global_best_pos - b_y
            upper_group = self.global_best_pos + b_y

            # intersect with domain
            lower_res = np.maximum(self.lower, lower_group)
            upper_res = np.minimum(self.upper, upper_group)

            # clip
            X_clipped[i] = np.clip(xi, lower_res, upper_res)

        return X_clipped

    def _random_skip(self, X):
        """
        Random Skip (eqs (20–22)):
            x_i = (x_i + r) / 2
        where r ~ U(lower, upper) component-wise.
        """
        k, d = X.shape
        rand_points = self.rng.uniform(self.lower, self.upper, size=(k, d))
        X_new = 0.5 * (X + rand_points)
        # also clip to bounds for safety
        return np.clip(X_new, self.lower, self.upper)

    def run(self, verbose=False):
        self._init_swarm()

        for it in range(self.max_iters):
            # 1) Position update
            X = self._position_update(self.positions)

            # 2) Evaluate fitness once after position update
            f_vals = np.apply_along_axis(self.func, 1, X)

            # 3) Check global + local improvements
            # Global improvements
            improved_global = f_vals < self.global_best_cost
            if np.any(improved_global):
                # best of the improved
                idx_candidates = np.where(improved_global)[0]
                idx_best = idx_candidates[np.argmin(f_vals[improved_global])]
                self.global_best_cost = float(f_vals[idx_best])
                self.global_best_pos = X[idx_best].copy()
                # halve global escape counter
                self.c_global = int(self.c_global / 2)

            # Local improvements (but not global)
            improved_local = (f_vals < self.local_best_cost) & (~improved_global)
            self.local_best_pos[improved_local] = X[improved_local]
            self.local_best_cost[improved_local] = f_vals[improved_local]
            # lower energy for improved locals (similar to eq (14) second case)
            self.energy[improved_local] = np.floor(self.energy[improved_local] / 2.0)

            # 4) Vibration for non-improved ones
            vib_mask = ~(improved_global | improved_local)
            X = self._vibration(X, vib_mask)

            # 5) Center clipping
            X = self._center_clipping(X)

            # 6) Update energy a_i (eqs (14–15))
            r4 = self.rng.uniform(0.0, 1.0, size=self.swarm_size)
            # increment energy when a_i * r4 < a_max
            inc_mask = (self.energy * r4) < self.a_max
            self.energy[inc_mask] += 1.0

            # 7) Random skip if global counter too large
            if self.c_global > self.c_max:
                X = self._random_skip(X)
                self.c_global = 0  # reset

            # commit positions
            self.positions = X

            # increment global counter
            self.c_global += 1

            if verbose and (it % max(1, self.max_iters // 10) == 0):
                print(f"iter={it:4d}, gbest={self.global_best_cost:.4e}, c={self.c_global}")

        return self.global_best_pos, self.global_best_cost


# ============================================================
# 3. Benchmark driver matching the paper's protocol
# ============================================================

DIM = 30
LOWER = -100.0
UPPER = 100.0
SWARM_SIZE = 100
MAX_ITERS = 1000
RUNS = 30  # paper uses 30 validations

results = []

for fname, f in benchmark_funcs:
    best_vals = []
    times = []
    print(f"Running HEO on {fname} ...")

    for run in range(RUNS):
        heo = HEO(
            func=f,
            dim=DIM,
            lower=LOWER,
            upper=UPPER,
            swarm_size=SWARM_SIZE,
            max_iters=MAX_ITERS,
            R=1.0,
            a_max=10.0,
            c_max=30,
            seed=run + 1234,  # reproducible runs
        )
        t0 = time.time()
        _, best_f = heo.run(verbose=False)
        t1 = time.time()

        best_vals.append(best_f)
        times.append(t1 - t0)

    best_vals = np.array(best_vals)
    times = np.array(times)

    mean_cost = float(np.mean(best_vals))
    std_cost = float(np.std(best_vals))
    mean_time_1000 = float(np.mean(times))  # seconds per 1000 iters (MAX_ITERS=1000)

    results.append({
        "Function": fname,
        "HEO_mean_cost": mean_cost,
        "HEO_std_cost": std_cost,
        "HEO_time_s_per_1000iters": mean_time_1000,
    })

df_results = pd.DataFrame(results)
display(df_results)


In [ ]:
import numpy as np
import pandas as pd
import time

# ============================================================
# 1. Benchmark functions (14, 30D, [-100,100])
# ============================================================

def sphere(x):
    return np.sum(x**2)

def step(x):
    return np.sum(np.floor(x + 0.5)**2)

def schwefel_221(x):
    return np.max(np.abs(x))

def schwefel_222(x):
    a = np.abs(x)
    return np.sum(a) + np.prod(a)

def rosenbrock(x):
    return np.sum(100.0 * (x[1:] - x[:-1]**2)**2 + (1 - x[:-1])**2)

def bent_cigar(x):
    return x[0]**2 + 1e6 * np.sum(x[1:]**2)

def sumsquares2(x):
    i = np.arange(1, x.size+1)
    return np.sum(i * x**2)

def alpine(x):
    return np.sum(np.abs(x * np.sin(x) + 0.1 * x))

def griewank(x):
    i = np.arange(1, x.size+1)
    return np.sum(x**2) / 4000.0 - np.prod(np.cos(x / np.sqrt(i))) + 1.0

def rastrigin(x):
    return 10.0 * x.size + np.sum(x**2 - 10.0 * np.cos(2*np.pi*x))

def ackley(x):
    d = x.size
    a = 20.0
    b = 0.2
    c = 2*np.pi
    s1 = np.sum(x**2)
    s2 = np.sum(np.cos(c*x))
    term1 = -a * np.exp(-b * np.sqrt(s1/d))
    term2 = -np.exp(s2/d)
    return term1 + term2 + a + np.e

def levy(x):
    w = 1 + (x - 1)/4
    term1 = np.sin(np.pi*w[0])**2
    term3 = (w[-1] - 1)**2 * (1 + np.sin(2*np.pi*w[-1])**2)
    term2 = np.sum((w[:-1] - 1)**2 * (1 + 10*np.sin(np.pi*w[:-1] + 1)**2))
    return term1 + term2 + term3

def salomon(x):
    r = np.sqrt(np.sum(x**2))
    return 1 - np.cos(2*np.pi*r) + 0.1*r

def schaffer(x):
    total = 0.0
    for i in range(x.size - 1):
        xi = x[i]
        xj = x[i+1]
        num = np.sin(np.sqrt(xi**2 + xj**2))**2 - 0.5
        den = (1 + 0.001*(xi**2 + xj**2))**2
        total += 0.5 + num/den
    return total

benchmark_funcs = [
    ("Sphere", sphere),
    ("Step", step),
    ("Schwefel 2.21", schwefel_221),
    ("Schwefel 2.22", schwefel_222),
    ("Rosenbrock", rosenbrock),
    ("BentCigar", bent_cigar),
    ("Sumsquares2", sumsquares2),
    ("Alpine", alpine),
    ("Griewank", griewank),
    ("Rastrigin", rastrigin),
    ("Ackley", ackley),
    ("Levy", levy),
    ("Salomon", salomon),
    ("Schaffer", schaffer),
]

# ============================================================
# 2. HEO-inspired optimizer (stable, HEO-style behaviour)
# ============================================================

class HEOInspired:
    """
    HEO-inspired optimizer:
      - Halfway move between personal and global best
      - PSO-like attraction to pbest and gbest
      - Vibration for stagnant particles
      - Center clipping around global best
      - Random skip when global stagnation is large
    Continuous minimization on [lower, upper]^dim.
    """
    def __init__(
        self,
        func,
        dim,
        lower=-100.0,
        upper=100.0,
        swarm_size=100,
        max_iters=1000,
        seed=None,
        # exploration/exploitation
        phi_p_max=2.5,
        phi_p_min=0.5,
        phi_g_max=2.5,
        phi_g_min=0.5,
        phi_h_max=1.5,
        phi_h_min=0.3,
        # stagnation control
        stagnation_patience_local=20,
        stagnation_patience_global=40,
        skip_patience=80,
        # vibration / clipping
        vibration_scale=0.2,
        clip_factor=2.0,
        skip_fraction=0.3,
    ):
        self.func = func
        self.dim = dim
        self.swarm_size = swarm_size
        self.max_iters = max_iters
        self.vibration_scale = vibration_scale
        self.clip_factor = clip_factor
        self.skip_fraction = skip_fraction

        if np.isscalar(lower):
            self.lower = np.full(dim, float(lower))
        else:
            self.lower = np.array(lower, dtype=float)

        if np.isscalar(upper):
            self.upper = np.full(dim, float(upper))
        else:
            self.upper = np.array(upper, dtype=float)

        self.rng = np.random.default_rng(seed)

        # coefficients schedule
        self.phi_p_max = phi_p_max
        self.phi_p_min = phi_p_min
        self.phi_g_max = phi_g_max
        self.phi_g_min = phi_g_min
        self.phi_h_max = phi_h_max
        self.phi_h_min = phi_h_min

        self.stagnation_patience_local = stagnation_patience_local
        self.stagnation_patience_global = stagnation_patience_global
        self.skip_patience = skip_patience

        # internal state
        self.X = None
        self.pbest = None
        self.pbest_f = None
        self.gbest = None
        self.gbest_f = None
        self.no_improve_local = None
        self.no_improve_global = 0

    def _init_swarm(self):
        self.X = self.rng.uniform(
            self.lower, self.upper, size=(self.swarm_size, self.dim)
        )
        self.pbest = self.X.copy()
        self.pbest_f = np.apply_along_axis(self.func, 1, self.pbest)
        idx = np.argmin(self.pbest_f)
        self.gbest = self.pbest[idx].copy()
        self.gbest_f = float(self.pbest_f[idx])
        self.no_improve_local = np.zeros(self.swarm_size, dtype=int)
        self.no_improve_global = 0

    def _coeff_schedule(self, t):
        """Linear decay of phi_p, phi_g, phi_h over iterations."""
        T = max(1, self.max_iters - 1)
        alpha = t / T
        phi_p = self.phi_p_max - alpha * (self.phi_p_max - self.phi_p_min)
        phi_g = self.phi_g_max - alpha * (self.phi_g_max - self.phi_g_min)
        phi_h = self.phi_h_max - alpha * (self.phi_h_max - self.phi_h_min)
        return phi_p, phi_g, phi_h

    def _position_update(self, t):
        """PSO-like + halfway move; stable step sizes."""
        phi_p, phi_g, phi_h = self._coeff_schedule(t)
        r_p = self.rng.random((self.swarm_size, self.dim))
        r_g = self.rng.random((self.swarm_size, self.dim))
        r_h = self.rng.random((self.swarm_size, self.dim))

        # broadcast global best
        g_mat = np.broadcast_to(self.gbest, (self.swarm_size, self.dim))
        p_mat = self.pbest

        halfway = 0.5 * (p_mat + g_mat)

        # main update
        step_p = phi_p * r_p * (p_mat - self.X)
        step_g = phi_g * r_g * (g_mat - self.X)
        step_h = phi_h * r_h * (halfway - self.X)

        X_new = self.X + step_p + step_g + step_h

        # clip to global bounds
        X_new = np.clip(X_new, self.lower, self.upper)
        return X_new

    def _vibration(self, X):
        """Shake particles that haven't improved locally for a while."""
        stagnant = self.no_improve_local >= self.stagnation_patience_local
        if not np.any(stagnant):
            return X

        # global std to scale noise
        std_vec = np.std(X, axis=0)
        std_vec = np.where(std_vec == 0.0, 1e-12, std_vec)

        idxs = np.where(stagnant)[0]
        for i in idxs:
            noise = self.rng.normal(
                loc=0.0,
                scale=self.vibration_scale * std_vec,
                size=self.dim,
            )
            X[i] = X[i] + noise

        X = np.clip(X, self.lower, self.upper)
        return X

    def _center_clipping(self, X):
        """Restrict particles to a box around global best."""
        # approximate spread
        diffs = np.abs(X - self.gbest)
        spread = np.max(diffs, axis=0)  # per-dimension max
        # radius proportional to spread but not smaller than a bit of domain
        domain_span = self.upper - self.lower
        min_radius = 0.05 * domain_span
        radius = np.maximum(self.clip_factor * spread, min_radius)

        box_lower = np.maximum(self.gbest - radius, self.lower)
        box_upper = np.minimum(self.gbest + radius, self.upper)

        # clip each particle to this "group" box intersected with global bounds
        return np.clip(X, box_lower, box_upper)

    def _random_skip(self, X):
        """Randomly re-init a fraction of worst particles when stuck."""
        if self.no_improve_global < self.skip_patience:
            return X

        # number of particles to re-init
        k = int(self.skip_fraction * self.swarm_size)
        if k <= 0:
            return X

        # indices of worst k particles
        worst_idx = np.argsort(self.pbest_f)[-k:]
        # random around global best within domain
        span = (self.upper - self.lower)
        rand_center = self.gbest
        low = np.maximum(rand_center - span * 0.5, self.lower)
        high = np.minimum(rand_center + span * 0.5, self.upper)
        X[worst_idx] = self.rng.uniform(low, high, size=(k, self.dim))

        # reset their local bests
        self.pbest[worst_idx] = X[worst_idx]
        self.pbest_f[worst_idx] = np.apply_along_axis(self.func, 1, self.pbest[worst_idx])
        self.no_improve_local[worst_idx] = 0

        # reset global stagnation counter
        self.no_improve_global = 0

        return X

    def run(self, verbose=False):
        self._init_swarm()
        for t in range(self.max_iters):
            # 1) halfway/PSO-like update
            X_new = self._position_update(t)

            # 2) vibration for stagnant locals
            X_new = self._vibration(X_new)

            # 3) center clipping around global best
            X_new = self._center_clipping(X_new)

            # 4) evaluate
            f_vals = np.apply_along_axis(self.func, 1, X_new)

            # 5) update local bests
            improved_local = f_vals < self.pbest_f
            self.pbest[improved_local] = X_new[improved_local]
            self.pbest_f[improved_local] = f_vals[improved_local]
            self.no_improve_local[improved_local] = 0
            self.no_improve_local[~improved_local] += 1

            # 6) update global best
            idx = np.argmin(f_vals)
            best_f = f_vals[idx]
            if best_f + 1e-12 < self.gbest_f:
                self.gbest_f = float(best_f)
                self.gbest = X_new[idx].copy()
                self.no_improve_global = 0
            else:
                self.no_improve_global += 1

            # 7) random skip if heavily stuck
            X_new = self._random_skip(X_new)

            self.X = X_new

            if verbose and (t % max(1, self.max_iters // 10) == 0):
                print(f"iter={t:4d}, gbest={self.gbest_f:.4e}")

        return self.gbest, self.gbest_f

# ============================================================
# 3. Benchmark protocol (same structure as paper)
# ============================================================

DIM = 30
LOWER = -100.0
UPPER = 100.0
SWARM_SIZE = 100
MAX_ITERS = 1000
RUNS = 30   # you can set to 10 if Kaggle is too slow

results = []

for fname, f in benchmark_funcs:
    best_vals = []
    times = []
    print(f"Running HEO-inspired on {fname} ...")
    for run in range(RUNS):
        heo = HEOInspired(
            func=f,
            dim=DIM,
            lower=LOWER,
            upper=UPPER,
            swarm_size=SWARM_SIZE,
            max_iters=MAX_ITERS,
            seed=run + 1234,
        )
        t0 = time.time()
        _, best_f = heo.run(verbose=False)
        t1 = time.time()

        best_vals.append(best_f)
        times.append(t1 - t0)

    best_vals = np.array(best_vals)
    times = np.array(times)

    results.append({
        "Function": fname,
        "HEOInspired_mean_cost": float(best_vals.mean()),
        "HEOInspired_std_cost": float(best_vals.std()),
        "HEOInspired_time_s_per_1000iters": float(times.mean()),
    })

df_results = pd.DataFrame(results)
display(df_results)


In [ ]:
import numpy as np
import pandas as pd
import time

# ============================================================
# 1. Benchmark functions (unimodal + multimodal)
# ============================================================

def sphere(x):
    return np.sum(x**2)

def step(x):
    return np.sum(np.floor(x + 0.5)**2)

def schwefel_221(x):
    return np.max(np.abs(x))

def schwefel_222(x):
    a = np.abs(x)
    return np.sum(a) + np.prod(a)

def rosenbrock(x):
    return np.sum(100.0 * (x[1:] - x[:-1]**2)**2 + (1 - x[:-1])**2)

def bent_cigar(x):
    return x[0]**2 + 1e6 * np.sum(x[1:]**2)

def sumsquares2(x):
    i = np.arange(1, x.size+1)
    return np.sum(i * x**2)

def alpine(x):
    return np.sum(np.abs(x * np.sin(x) + 0.1 * x))

def griewank(x):
    i = np.arange(1, x.size+1)
    return np.sum(x**2) / 4000.0 - np.prod(np.cos(x / np.sqrt(i))) + 1.0

def rastrigin(x):
    return 10.0 * x.size + np.sum(x**2 - 10.0 * np.cos(2*np.pi*x))

def ackley(x):
    d = x.size
    a = 20.0
    b = 0.2
    c = 2*np.pi
    s1 = np.sum(x**2)
    s2 = np.sum(np.cos(c*x))
    term1 = -a * np.exp(-b * np.sqrt(s1/d))
    term2 = -np.exp(s2/d)
    return term1 + term2 + a + np.e

def levy(x):
    w = 1 + (x - 1)/4
    term1 = np.sin(np.pi*w[0])**2
    term3 = (w[-1] - 1)**2 * (1 + np.sin(2*np.pi*w[-1])**2)
    term2 = np.sum((w[:-1] - 1)**2 * (1 + 10*np.sin(np.pi*w[:-1] + 1)**2))
    return term1 + term2 + term3

def salomon(x):
    r = np.sqrt(np.sum(x**2))
    return 1 - np.cos(2*np.pi*r) + 0.1*r

def schaffer(x):
    total = 0.0
    for i in range(x.size - 1):
        xi = x[i]
        xj = x[i+1]
        num = np.sin(np.sqrt(xi**2 + xj**2))**2 - 0.5
        den = (1 + 0.001*(xi**2 + xj**2))**2
        total += 0.5 + num/den
    return total

benchmark_funcs = [
    ("Sphere", sphere),
    ("Step", step),
    ("Schwefel 2.21", schwefel_221),
    ("Schwefel 2.22", schwefel_222),
    ("Rosenbrock", rosenbrock),
    ("BentCigar", bent_cigar),
    ("Sumsquares2", sumsquares2),
    ("Alpine", alpine),
    ("Griewank", griewank),
    ("Rastrigin", rastrigin),
    ("Ackley", ackley),
    ("Levy", levy),
    ("Salomon", salomon),
    ("Schaffer", schaffer),
]

# ============================================================
# 2. HEO implementation (Algorithm 1 + eqs (1)-(7),(13)-(22))
# ============================================================

class HEO:
    """
    Halfway Escape Optimization (HEO) as literally as possible from:

      Algorithm 1 (p.7)
      Position update eqs (1)-(3), random r1,r2,r3 eqs (4)-(6), escape c_i eq (7)
      Vibration eqs (13)-(16)
      Center clipping eqs (17)-(19)
      Random skip eqs (20)-(22)

    Assumptions where the paper is ambiguous are kept minimal and commented.
    """

    def __init__(
        self,
        func,
        dim,
        bound=100.0,
        swarm_size=100,
        max_iters=1000,
        a_max=10,
        c_max=30,
        R=1.0,
        seed=None,
    ):
        self.func = func
        self.dim = dim
        self.bound = float(bound)
        self.swarm_size = swarm_size
        self.max_iters = max_iters
        self.a_max = float(a_max)
        self.c_max = int(c_max)
        self.R = float(R)

        self.lower = -self.bound * np.ones(dim)
        self.upper = self.bound * np.ones(dim)

        self.rng = np.random.default_rng(seed)

        # Swarm state
        self.X = None               # positions
        self.local_best = None      # personal best positions
        self.local_best_f = None    # personal best fitness
        self.a = None               # energy levels a_i per quantum
        self.c = 0                  # escape counter c_i (global)
        self.global_best = None
        self.global_best_f = None

    def _init_swarm(self):
        # Positions uniform in [-bound, bound]^dim
        self.X = self.rng.uniform(self.lower, self.upper,
                                  size=(self.swarm_size, self.dim))
        # Initial personal bests
        self.local_best = self.X.copy()
        self.local_best_f = np.apply_along_axis(self.func, 1, self.local_best)

        # Global best
        idx = np.argmin(self.local_best_f)
        self.global_best = self.local_best[idx].copy()
        self.global_best_f = float(self.local_best_f[idx])

        # Energy levels start at 0
        self.a = np.zeros(self.swarm_size, dtype=float)

        # Escape counter c_i = 0
        self.c = 0

    def _position_update_single(self, x, x_l):
        """
        Position update for one quantum, using eqs. (1)-(3):

          x_{i+1} = x_i + v_g + v_l
          v_g = (x_g - x_i (c_i + 1) * r1) * r2 * r3
          v_l = (x_l - x_i (c_i + 1) * r1) * r2 * (1 - r3)

        We treat r1,r2,r3 as vectors in R^dim, drawn per quantum per iteration.
        """
        x_g = self.global_best

        r1 = self.rng.uniform(1.0 - self.R, 1.0 + self.R, size=self.dim)  # eq (4)
        r2 = self.rng.uniform(0.5, 1.5, size=self.dim)                    # eq (5)
        r3 = self.rng.uniform(0.0, 1.0, size=self.dim)                    # eq (6)

        # term x_i (c_i+1) * r1
        term = x * ((self.c + 1.0) * r1)

        v_g = (x_g - term) * r2 * r3
        v_l = (x_l - term) * r2 * (1.0 - r3)

        x_new = x + v_g + v_l
        # Always enforce search bound S_bound
        x_new = np.clip(x_new, self.lower, self.upper)
        return x_new

    def _vibration(self, x_current, sigma_vec, a_i):
        """
        Vibration step for one quantum (eq. (13)-(16)):

          x_{i+1} = x_i + n / (1 + e^{a_i}), where n ~ N(0, σ_x)
        """
        # n ~ N(0, σ_x) component-wise
        n = self.rng.normal(loc=0.0, scale=sigma_vec, size=self.dim)
        scale = 1.0 / (1.0 + np.exp(a_i))
        x_new = x_current + scale * n
        x_new = np.clip(x_new, self.lower, self.upper)
        return x_new

    def _center_clipping(self, x_current):
        """
        Center clipping (eq. (17)-(19)):

          S_result = S_bound ∩ S_group
          b_y = ||x_i - x_g||_2 * r5,   r5 ~ U(0,2)
          S_group: hypercube centered at x_g with half-side-length b_y
        """
        diff = x_current - self.global_best
        dist = np.linalg.norm(diff, ord=2)
        r5 = self.rng.uniform(0.0, 2.0)  # eq. (19)
        b_y = dist * r5                  # eq. (18)

        lower_group = self.global_best - b_y
        upper_group = self.global_best + b_y

        lower_res = np.maximum(self.lower, lower_group)
        upper_res = np.minimum(self.upper, upper_group)

        x_clipped = np.clip(x_current, lower_res, upper_res)
        return x_clipped

    def _random_skip_all(self):
        """
        Random skip (eq. (20)-(22)), applied to ALL quantums when c_i > c_max:

          x_i = (x_i + r_vec) / 2
          r_j ~ U(0, b),  b = bound
        """
        b = self.bound
        rand_vecs = self.rng.uniform(0.0, b, size=(self.swarm_size, self.dim))
        self.X = 0.5 * (self.X + rand_vecs)
        self.X = np.clip(self.X, self.lower, self.upper)
        # Typically leave personal/global bests; paper says "group random skip"
        # Algorithm 1 resets c_i to 0
        self.c = 0

    def run(self, verbose=False):
        self._init_swarm()

        for it in range(self.max_iters):
            # Compute standard deviation of positions for vibration step
            sigma_vec = np.std(self.X, axis=0)
            sigma_vec = np.where(sigma_vec == 0.0, 1e-12, sigma_vec)

            # Work with a copy for consistent "x_i" in this iteration
            X_old = self.X.copy()

            for j in range(self.swarm_size):
                x_old = X_old[j]
                x_l   = self.local_best[j]
                a_i   = self.a[j]

                # --- Position update (eqs (1)-(3)) ---
                x_pu = self._position_update_single(x_old, x_l)
                f_q  = self.func(x_pu)

                # --- Global best update (Algorithm 1) ---
                if f_q < self.global_best_f:
                    self.global_best_f = float(f_q)
                    self.global_best   = x_pu.copy()
                    self.X[j]          = x_pu
                    self.local_best[j] = x_pu
                    self.local_best_f[j] = f_q
                    # c_i <- int(c_i / 2)
                    self.c = int(self.c / 2)

                # --- Local best update (Algorithm 1) ---
                elif f_q < self.local_best_f[j]:
                    self.local_best[j] = x_pu
                    self.local_best_f[j] = f_q
                    self.X[j] = x_pu
                    # a_i <- int(a_i / 2)
                    self.a[j] = float(int(a_i / 2))

                else:
                    # --- Vibration (eq. (13)-(16)) ---
                    x_vib = self._vibration(x_pu, sigma_vec, a_i)
                    self.X[j] = x_vib

                # --- Center clipping (eq. (17)-(19)) ---
                self.X[j] = self._center_clipping(self.X[j])

                # --- Energy level increment (eq. (14)-(15)) ---
                r4 = self.rng.uniform(0.0, 1.0)  # eq. (15)
                # Only increment if a_i * r4 < a_max
                if self.a[j] * r4 < self.a_max:
                    self.a[j] += 1.0

            # --- Random skip if c_i > c_max (Algorithm 1 + eq. (20)-(22)) ---
            if self.c > self.c_max:
                self._random_skip_all()

            # global escape counter increments each iteration (Algorithm 1, c_i++)
            self.c += 1

            if verbose and (it % max(1, self.max_iters // 10) == 0):
                print(f"iter={it:4d}, gbest={self.global_best_f:.4e}, c={self.c}")

        return self.global_best, self.global_best_f

# ============================================================
# 3. Benchmark: 14 functions, dim=30, bound=100, swarm=100, iters=1000
# ============================================================

DIM = 30
BOUND = 100.0
SWARM_SIZE = 100
MAX_ITERS = 1000
RUNS = 30   # can reduce to 10 if this is too slow

results = []

for fname, f in benchmark_funcs:
    best_vals = []
    times = []
    print(f"Running HEO (paper-style) on {fname} ...")
    for run in range(RUNS):
        heo = HEO(
            func=f,
            dim=DIM,
            bound=BOUND,
            swarm_size=SWARM_SIZE,
            max_iters=MAX_ITERS,
            a_max=10,
            c_max=30,
            R=1.0,
            seed=run + 1234,
        )
        t0 = time.time()
        _, best_f = heo.run(verbose=False)
        t1 = time.time()

        best_vals.append(best_f)
        times.append(t1 - t0)

    best_vals = np.array(best_vals)
    times = np.array(times)

    results.append({
        "Function": fname,
        "HEO_mean_cost": float(best_vals.mean()),
        "HEO_std_cost": float(best_vals.std()),
        "HEO_time_s_per_1000iters": float(times.mean()),
    })

df_results = pd.DataFrame(results)
display(df_results)


In [ ]:
import numpy as np
import pandas as pd
import time

# ============================================================
# 1. Benchmark functions (unimodal + multimodal)
# ============================================================

def sphere(x):
    return np.sum(x**2)

def step(x):
    return np.sum(np.floor(x + 0.5)**2)

def schwefel_221(x):
    return np.max(np.abs(x))

def schwefel_222(x):
    a = np.abs(x)
    return np.sum(a) + np.prod(a)

def rosenbrock(x):
    return np.sum(100.0 * (x[1:] - x[:-1]**2)**2 + (1 - x[:-1])**2)

def bent_cigar(x):
    return x[0]**2 + 1e6 * np.sum(x[1:]**2)

def sumsquares2(x):
    i = np.arange(1, x.size+1)
    return np.sum(i * x**2)

def alpine(x):
    return np.sum(np.abs(x * np.sin(x) + 0.1 * x))

def griewank(x):
    i = np.arange(1, x.size+1)
    return np.sum(x**2) / 4000.0 - np.prod(np.cos(x / np.sqrt(i))) + 1.0

def rastrigin(x):
    return 10.0 * x.size + np.sum(x**2 - 10.0 * np.cos(2*np.pi*x))

def ackley(x):
    d = x.size
    a = 20.0
    b = 0.2
    c = 2*np.pi
    s1 = np.sum(x**2)
    s2 = np.sum(np.cos(c*x))
    term1 = -a * np.exp(-b * np.sqrt(s1/d))
    term2 = -np.exp(s2/d)
    return term1 + term2 + a + np.e

def levy(x):
    w = 1 + (x - 1)/4
    term1 = np.sin(np.pi*w[0])**2
    term3 = (w[-1] - 1)**2 * (1 + np.sin(2*np.pi*w[-1])**2)
    term2 = np.sum((w[:-1] - 1)**2 * (1 + 10*np.sin(np.pi*w[:-1] + 1)**2))
    return term1 + term2 + term3

def salomon(x):
    r = np.sqrt(np.sum(x**2))
    return 1 - np.cos(2*np.pi*r) + 0.1*r

def schaffer(x):
    total = 0.0
    for i in range(x.size - 1):
        xi = x[i]
        xj = x[i+1]
        num = np.sin(np.sqrt(xi**2 + xj**2))**2 - 0.5
        den = (1 + 0.001*(xi**2 + xj**2))**2
        total += 0.5 + num/den
    return total

benchmark_funcs = [
    ("Sphere", sphere),
    ("Step", step),
    ("Schwefel 2.21", schwefel_221),
    ("Schwefel 2.22", schwefel_222),
    ("Rosenbrock", rosenbrock),
    ("BentCigar", bent_cigar),
    ("Sumsquares2", sumsquares2),
    ("Alpine", alpine),
    ("Griewank", griewank),
    ("Rastrigin", rastrigin),
    ("Ackley", ackley),
    ("Levy", levy),
    ("Salomon", salomon),
    ("Schaffer", schaffer),
]

# ============================================================
# 2. HEO implementation (Algorithm 1 + eqs (1)-(7),(13)-(22))
# ============================================================

class HEO:
    """
    Halfway Escape Optimization (HEO) as literally as possible from:

      Algorithm 1 (p.7)
      Position update eqs (1)-(3), random r1,r2,r3 eqs (4)-(6), escape c_i eq (7)
      Vibration eqs (13)-(16)
      Center clipping eqs (17)-(19)
      Random skip eqs (20)-(22)

    Assumptions where the paper is ambiguous are kept minimal and commented.
    """

    def __init__(
        self,
        func,
        dim,
        bound=100.0,
        swarm_size=100,
        max_iters=1000,
        a_max=10,
        c_max=30,
        R=1.0,
        seed=None,
    ):
        self.func = func
        self.dim = dim
        self.bound = float(bound)
        self.swarm_size = swarm_size
        self.max_iters = max_iters
        self.a_max = float(a_max)
        self.c_max = int(c_max)
        self.R = float(R)

        self.lower = -self.bound * np.ones(dim)
        self.upper = self.bound * np.ones(dim)

        self.rng = np.random.default_rng(seed)

        # Swarm state
        self.X = None               # positions
        self.local_best = None      # personal best positions
        self.local_best_f = None    # personal best fitness
        self.a = None               # energy levels a_i per quantum
        self.c = None               # escape counter for each quantum
        self.global_best = None
        self.global_best_f = None

    def _init_swarm(self):
        # Positions uniform in [-bound, bound]^dim
        self.X = self.rng.uniform(self.lower, self.upper,
                                  size=(self.swarm_size, self.dim))
        # Initial personal bests
        self.local_best = self.X.copy()
        self.local_best_f = np.apply_along_axis(self.func, 1, self.local_best)

        # Global best
        idx = np.argmin(self.local_best_f)
        self.global_best = self.local_best[idx].copy()
        self.global_best_f = float(self.local_best_f[idx])

        # Energy levels start at 0
        self.a = np.zeros(self.swarm_size, dtype=float)

        # Escape counter c_i = 0 for each quantum
        self.c = np.zeros(self.swarm_size, dtype=int)

    def _position_update_single(self, x, x_l):
        """
        Position update for one quantum, using eqs. (1)-(3):

          x_{i+1} = x_i + v_g + v_l
          v_g = (x_g - x_i (c_i + 1) * r1) * r2 * r3
          v_l = (x_l - x_i (c_i + 1) * r1) * r2 * (1 - r3)

        We treat r1,r2,r3 as vectors in R^dim, drawn per quantum per iteration.
        """
        x_g = self.global_best

        r1 = self.rng.uniform(1.0 - self.R, 1.0 + self.R, size=self.dim)  # eq (4)
        r2 = self.rng.uniform(0.5, 1.5, size=self.dim)                    # eq (5)
        r3 = self.rng.uniform(0.0, 1.0, size=self.dim)                    # eq (6)

        # term x_i (c_i+1) * r1
        term = x * ((self.c[0] + 1.0) * r1)

        v_g = (x_g - term) * r2 * r3
        v_l = (x_l - term) * r2 * (1.0 - r3)

        x_new = x + v_g + v_l
        # Always enforce search bound S_bound
        x_new = np.clip(x_new, self.lower, self.upper)
        return x_new

    def _vibration(self, x_current, sigma_vec, a_i):
        """
        Vibration step for one quantum (eq. (13)-(16)):

          x_{i+1} = x_i + n / (1 + e^{a_i}), where n ~ N(0, σ_x)
        """
        # n ~ N(0, σ_x) component-wise
        n = self.rng.normal(loc=0.0, scale=sigma_vec, size=self.dim)
        scale = 1.0 / (1.0 + np.exp(a_i))
        x_new = x_current + scale * n
        x_new = np.clip(x_new, self.lower, self.upper)
        return x_new

    def _center_clipping(self, x_current):
        """
        Center clipping (eq. (17)-(19)):

          S_result = S_bound ∩ S_group
          b_y = ||x_i - x_g||_2 * r5,   r5 ~ U(0,2)
          S_group: hypercube centered at x_g with half-side-length b_y
        """
        diff = x_current - self.global_best
        dist = np.linalg.norm(diff, ord=2)
        r5 = self.rng.uniform(0.0, 2.0)  # eq. (19)
        b_y = dist * r5                  # eq. (18)

        lower_group = self.global_best - b_y
        upper_group = self.global_best + b_y

        lower_res = np.maximum(self.lower, lower_group)
        upper_res = np.minimum(self.upper, upper_group)

        x_clipped = np.clip(x_current, lower_res, upper_res)
        return x_clipped

    def _random_skip_all(self):
        """
        Random skip (eq. (20)-(22)), applied to ALL quantums when c_i > c_max:

          x_i = (x_i + r_vec) / 2
          r_j ~ U(0, b),  b = bound
        """
        b = self.bound
        rand_vecs = self.rng.uniform(0.0, b, size=(self.swarm_size, self.dim))
        self.X = 0.5 * (self.X + rand_vecs)
        self.X = np.clip(self.X, self.lower, self.upper)
        # Typically leave personal/global bests; paper says "group random skip"
        # Algorithm 1 resets c_i to 0
        self.c = np.zeros(self.swarm_size, dtype=int)

    def run(self, verbose=False):
        self._init_swarm()

        for it in range(self.max_iters):
            # Compute standard deviation of positions for vibration step
            sigma_vec = np.std(self.X, axis=0)
            sigma_vec = np.where(sigma_vec == 0.0, 1e-12, sigma_vec)

            # Work with a copy for consistent "x_i" in this iteration
            X_old = self.X.copy()

            for j in range(self.swarm_size):
                x_old = X_old[j]
                x_l   = self.local_best[j]
                a_i   = self.a[j]

                # --- Position update (eqs (1)-(3)) ---
                x_pu = self._position_update_single(x_old, x_l)
                f_q  = self.func(x_pu)

                # --- Global best update (Algorithm 1) ---
                if f_q < self.global_best_f:
                    self.global_best_f = float(f_q)
                    self.global_best   = x_pu.copy()
                    self.X[j]          = x_pu
                    self.local_best[j] = x_pu
                    self.local_best_f[j] = f_q
                    # c_i <- int(c_i / 2)
                    self.c[j] = int(self.c[j] / 2)

                # --- Local best update (Algorithm 1) ---
                elif f_q < self.local_best_f[j]:
                    self.local_best[j] = x_pu
                    self.local_best_f[j] = f_q
                    self.X[j] = x_pu
                    # a_i <- int(a_i / 2)
                    self.a[j] = float(int(a_i / 2))

                else:
                    # --- Vibration (eq. (13)-(16)) ---
                    x_vib = self._vibration(x_pu, sigma_vec, a_i)
                    self.X[j] = x_vib

                # --- Center clipping (eq. (17)-(19)) ---
                self.X[j] = self._center_clipping(self.X[j])

                # --- Energy level increment (eq. (14)-(15)) ---
                r4 = self.rng.uniform(0.0, 1.0)  # eq. (15)
                # Only increment if a_i * r4 < a_max
                if self.a[j] * r4 < self.a_max:
                    self.a[j] += 1.0

            # --- Random skip if c_i > c_max (Algorithm 1 + eq. (20)-(22)) ---
            if np.any(self.c > self.c_max):
                self._random_skip_all()

            # global escape counter increments each iteration (Algorithm 1, c_i++)
            self.c += 1

            if verbose and (it % max(1, self.max_iters // 10) == 0):
                print(f"iter={it:4d}, gbest={self.global_best_f:.4e}, c={self.c[0]}")

        return self.global_best, self.global_best_f

# ============================================================
# 3. Benchmark: 14 functions, dim=30, bound=100, swarm=100, iters=1000
# ============================================================

DIM = 30
BOUND = 100.0
SWARM_SIZE = 100
MAX_ITERS = 1000
RUNS = 30   # can reduce to 10 if this is too slow

results = []

for fname, f in benchmark_funcs:
    best_vals = []
    times = []
    print(f"Running HEO (paper-style) on {fname} ...")
    for run in range(RUNS):
        heo = HEO(
            func=f,
            dim=DIM,
            bound=BOUND,
            swarm_size=SWARM_SIZE,
            max_iters=MAX_ITERS,
            a_max=10,
            c_max=30,
            R=1.0,
            seed=run + 1234,
        )
        t0 = time.time()
        _, best_f = heo.run(verbose=False)
        t1 = time.time()

        best_vals.append(best_f)
        times.append(t1 - t0)

    best_vals = np.array(best_vals)
    times = np.array(times)

    results.append({
        "Function": fname,
        "HEO_mean_cost": float(best_vals.mean()),
        "HEO_std_cost": float(best_vals.std()),
        "HEO_time_s_per_1000iters": float(times.mean()),
    })

df_results = pd.DataFrame(results)
display(df_results)


Running HEO (paper-style) on Sphere ...
Running HEO (paper-style) on Step ...
Running HEO (paper-style) on Schwefel 2.21 ...
Running HEO (paper-style) on Schwefel 2.22 ...
Running HEO (paper-style) on Rosenbrock ...
Running HEO (paper-style) on BentCigar ...
Running HEO (paper-style) on Sumsquares2 ...
Running HEO (paper-style) on Alpine ...
Running HEO (paper-style) on Griewank ...
Running HEO (paper-style) on Rastrigin ...
Running HEO (paper-style) on Ackley ...
Running HEO (paper-style) on Levy ...
Running HEO (paper-style) on Salomon ...
Running HEO (paper-style) on Schaffer ...


In [ ]:
import numpy as np
import pandas as pd
import time

# ============================================================
# 1) Benchmark functions (same list you used)
# ============================================================
def sphere(x): return np.sum(x**2)
def step(x): return np.sum(np.floor(x + 0.5)**2)
def schwefel_221(x): return np.max(np.abs(x))
def schwefel_222(x):
    a = np.abs(x)
    return np.sum(a) + np.prod(a)
def rosenbrock(x): return np.sum(100.0*(x[1:] - x[:-1]**2)**2 + (1-x[:-1])**2)
def bent_cigar(x): return x[0]**2 + 1e6*np.sum(x[1:]**2)
def sumsquares2(x):
    i = np.arange(1, x.size+1)
    return np.sum(i * x**2)
def alpine(x): return np.sum(np.abs(x*np.sin(x) + 0.1*x))
def griewank(x):
    i = np.arange(1, x.size+1)
    return np.sum(x**2)/4000.0 - np.prod(np.cos(x/np.sqrt(i))) + 1.0
def rastrigin(x): return 10.0*x.size + np.sum(x**2 - 10.0*np.cos(2*np.pi*x))
def ackley(x):
    d = x.size
    a, b, c = 20.0, 0.2, 2*np.pi
    s1 = np.sum(x**2)
    s2 = np.sum(np.cos(c*x))
    return (-a*np.exp(-b*np.sqrt(s1/d)) - np.exp(s2/d) + a + np.e)
def levy(x):
    w = 1 + (x - 1)/4
    term1 = np.sin(np.pi*w[0])**2
    term3 = (w[-1]-1)**2 * (1 + np.sin(2*np.pi*w[-1])**2)
    term2 = np.sum((w[:-1]-1)**2 * (1 + 10*np.sin(np.pi*w[:-1] + 1)**2))
    return term1 + term2 + term3
def salomon(x):
    r = np.sqrt(np.sum(x**2))
    return 1 - np.cos(2*np.pi*r) + 0.1*r
def schaffer(x):
    total = 0.0
    for i in range(x.size - 1):
        xi, xj = x[i], x[i+1]
        num = np.sin(np.sqrt(xi**2 + xj**2))**2 - 0.5
        den = (1 + 0.001*(xi**2 + xj**2))**2
        total += 0.5 + num/den
    return total

benchmark_funcs = [
    ("Sphere", sphere),
    ("Step", step),
    ("Schwefel 2.21", schwefel_221),
    ("Schwefel 2.22", schwefel_222),
    ("Rosenbrock", rosenbrock),
    ("BentCigar", bent_cigar),
    ("Sumsquares2", sumsquares2),
    ("Alpine", alpine),
    ("Griewank", griewank),
    ("Rastrigin", rastrigin),
    ("Ackley", ackley),
    ("Levy", levy),
    ("Salomon", salomon),
    ("Schaffer", schaffer),
]

# ============================================================
# 2) Modified HEO + (1+1)-ES (your requested changes)
#    - remove r2
#    - annealed vibration -> 0
#    - add 1/5 success rule ES local search
# ============================================================
class HEO_ES:
    def __init__(
        self,
        func,
        dim,
        bound=100.0,
        swarm_size=100,
        max_iters=1000,
        c_max=30,
        R=1.0,
        # vibration
        vib_sigma0=0.1,      # base noise scale (multiplied by anneal(t))
        vib_schedule="exp",  # "exp" or "linear"
        # ES
        es_init_sigma=10.0,  # per-particle sigma initial
        es_window=20,        # trials window for success rate
        es_c=0.85,            # 1/5-rule multiplier
        seed=None
    ):
        self.func = func
        self.dim = int(dim)
        self.bound = float(bound)
        self.N = int(swarm_size)
        self.T = int(max_iters)
        self.c_max = int(c_max)
        self.R = float(R)
        self.vib_sigma0 = float(vib_sigma0)
        self.vib_schedule = vib_schedule

        self.es_init_sigma = float(es_init_sigma)
        self.es_window = int(es_window)
        self.es_c = float(es_c)

        self.lower = -self.bound * np.ones(self.dim)
        self.upper =  self.bound * np.ones(self.dim)
        self.rng = np.random.default_rng(seed)

    def _anneal(self, t):
        if self.vib_schedule == "linear":
            return max(0.0, 1.0 - (t / max(1, self.T - 1)))
        # default: exp
        return float(np.exp(-t / max(1, self.T)))

    def run(self, verbose=False):
        # init swarm
        X = self.rng.uniform(self.lower, self.upper, size=(self.N, self.dim))
        f = np.array([self.func(x) for x in X], dtype=float)

        pbest = X.copy()
        pbest_f = f.copy()

        gidx = int(np.argmin(f))
        gbest = X[gidx].copy()
        gbest_f = float(f[gidx])

        # per-particle escape counters
        c = np.zeros(self.N, dtype=int)

        # per-particle ES sigma + success tracking
        sigma = np.full(self.N, self.es_init_sigma, dtype=float)
        succ = np.zeros(self.N, dtype=int)
        trials = np.zeros(self.N, dtype=int)

        for t in range(self.T):
            anneal = self._anneal(t)

            # optional: for vibration scaling with population spread (more adaptive)
            # (kept mild, still decays to 0)
            pop_std = np.std(X, axis=0)
            pop_std = np.where(pop_std == 0.0, 1e-12, pop_std)

            for i in range(self.N):
                x = X[i]

                # --- HEO update without r2 ---
                r1 = self.rng.uniform(1.0 - self.R, 1.0 + self.R, size=self.dim)
                r3 = self.rng.uniform(0.0, 1.0, size=self.dim)

                term = x * (c[i] + 1.0) * r1
                v_g  = (gbest - term) * r3
                v_l  = (pbest[i] - term) * (1.0 - r3)

                x_new = x + v_g + v_l

                # --- annealed vibration (goes to 0) ---
                # use pop_std for scale but multiplied by vib_sigma0 and anneal(t)
                x_new = x_new + (anneal * self.vib_sigma0) * (pop_std * self.rng.normal(0.0, 1.0, size=self.dim))

                # clamp to bounds
                x_new = np.clip(x_new, self.lower, self.upper)

                # --- (1+1)-ES local search ---
                x_mut = x_new + sigma[i] * self.rng.normal(0.0, 1.0, size=self.dim)
                x_mut = np.clip(x_mut, self.lower, self.upper)
                f_mut = float(self.func(x_mut))

                trials[i] += 1
                if f_mut < f[i]:
                    # accept
                    X[i] = x_mut
                    f[i] = f_mut
                    succ[i] += 1
                    c[i] = 0
                else:
                    # reject, keep old
                    c[i] += 1

                # update personal best
                if f[i] < pbest_f[i]:
                    pbest[i] = X[i].copy()
                    pbest_f[i] = f[i]

                # update global best
                if f[i] < gbest_f:
                    gbest = X[i].copy()
                    gbest_f = float(f[i])

                # --- 1/5 success rule every window ---
                if trials[i] >= self.es_window:
                    rate = succ[i] / trials[i]
                    if rate > 0.2:
                        sigma[i] /= self.es_c
                    else:
                        sigma[i] *= self.es_c
                    # prevent sigma going crazy
                    sigma[i] = float(np.clip(sigma[i], 1e-12, self.bound))
                    succ[i] = 0
                    trials[i] = 0

                # --- random skip if stagnating too long ---
                if c[i] > self.c_max:
                    X[i] = self.rng.uniform(self.lower, self.upper)
                    f[i] = float(self.func(X[i]))
                    c[i] = 0
                    # (optional) reset ES sigma a bit after skip
                    sigma[i] = self.es_init_sigma

            if verbose and (t % max(1, self.T // 10) == 0):
                print(f"iter={t:4d}, gbest={gbest_f:.6e}")

        return gbest, gbest_f

# ============================================================
# 3) Run the benchmark protocol and report table
# ============================================================
DIM = 30
BOUND = 100.0
SWARM_SIZE = 100
MAX_ITERS = 1000
RUNS = 30   # if too slow on Kaggle, set to 10 first

# params you asked for:
R = 1.0
C_MAX = 30

# vibration that can converge (anneals to ~0):
VIB_SIGMA0 = 0.10     # try 0.05..0.20
VIB_SCHEDULE = "exp"  # "exp" or "linear"

# ES params (1/5 success rule):
ES_INIT_SIGMA = 10.0  # try 5..20
ES_WINDOW = 20
ES_C = 0.85

results = []

for fname, func in benchmark_funcs:
    best_vals = []
    times = []
    print(f"Running Modified HEO (no r2) + annealed vibration + (1+1)-ES on {fname} ...")

    for run in range(RUNS):
        opt = HEO_ES(
            func=func,
            dim=DIM,
            bound=BOUND,
            swarm_size=SWARM_SIZE,
            max_iters=MAX_ITERS,
            c_max=C_MAX,
            R=R,
            vib_sigma0=VIB_SIGMA0,
            vib_schedule=VIB_SCHEDULE,
            es_init_sigma=ES_INIT_SIGMA,
            es_window=ES_WINDOW,
            es_c=ES_C,
            seed=run + 1234
        )

        t0 = time.time()
        _, best_f = opt.run(verbose=False)
        t1 = time.time()

        best_vals.append(best_f)
        times.append(t1 - t0)

    best_vals = np.array(best_vals, dtype=float)
    times = np.array(times, dtype=float)

    results.append({
        "Function": fname,
        "HEO+ES_mean_cost": float(best_vals.mean()),
        "HEO+ES_std_cost": float(best_vals.std()),
        "time_s_per_1000iters": float(times.mean()),
    })

df = pd.DataFrame(results)
display(df)

print("\nDone. If you want closer-to-zero results, try:")
print("- lower VIB_SIGMA0 (e.g., 0.05)")
print("- more iterations (e.g., 3000 or 5000)")
print("- slightly smaller ES_INIT_SIGMA if you see bouncing near optimum")


Running Modified HEO (no r2) + annealed vibration + (1+1)-ES on Sphere ...
Running Modified HEO (no r2) + annealed vibration + (1+1)-ES on Step ...
Running Modified HEO (no r2) + annealed vibration + (1+1)-ES on Schwefel 2.21 ...
Running Modified HEO (no r2) + annealed vibration + (1+1)-ES on Schwefel 2.22 ...
Running Modified HEO (no r2) + annealed vibration + (1+1)-ES on Rosenbrock ...
Running Modified HEO (no r2) + annealed vibration + (1+1)-ES on BentCigar ...
Running Modified HEO (no r2) + annealed vibration + (1+1)-ES on Sumsquares2 ...
Running Modified HEO (no r2) + annealed vibration + (1+1)-ES on Alpine ...
Running Modified HEO (no r2) + annealed vibration + (1+1)-ES on Griewank ...
Running Modified HEO (no r2) + annealed vibration + (1+1)-ES on Rastrigin ...
Running Modified HEO (no r2) + annealed vibration + (1+1)-ES on Ackley ...
Running Modified HEO (no r2) + annealed vibration + (1+1)-ES on Levy ...
Running Modified HEO (no r2) + annealed vibration + (1+1)-ES on Salomon .

,Function,HEO+ES_mean_cost,HEO+ES_std_cost,time_s_per_1000iters
0,Sphere,2.423575e+03,2.624858e+02,4.603642
1,Step,2.526567e+03,3.247749e+02,4.321736
2,Schwefel 2.21,3.072552e+01,4.317420e+00,4.028472
3,Schwefel 2.22,1.891714e+21,4.112356e+21,4.388983
4,Rosenbrock,5.857323e+07,1.879244e+07,4.833165
5,BentCigar,2.310047e+09,3.843669e+08,4.207525
6,Sumsquares2,3.517374e+04,5.833863e+03,4.451585
7,Alpine,1.442544e+02,1.492256e+01,4.467406
8,Griewank,1.605894e+00,6.562145e-02,5.193915
9,Rastrigin,2.778673e+03,2.664586e+02,4.595113



Done. If you want closer-to-zero results, try:
- lower VIB_SIGMA0 (e.g., 0.05)
- more iterations (e.g., 3000 or 5000)
- slightly smaller ES_INIT_SIGMA if you see bouncing near optimum


In [ ]:
import numpy as np
import pandas as pd
import time

# ============================================================
# 1) Benchmark functions
# ============================================================
def sphere(x): return np.sum(x**2)
def step(x): return np.sum(np.floor(x + 0.5)**2)
def schwefel_221(x): return np.max(np.abs(x))
def schwefel_222(x):
    a = np.abs(x)
    return np.sum(a) + np.prod(a)
def rosenbrock(x): return np.sum(100.0*(x[1:] - x[:-1]**2)**2 + (1-x[:-1])**2)
def bent_cigar(x): return x[0]**2 + 1e6*np.sum(x[1:]**2)
def sumsquares2(x):
    i = np.arange(1, x.size+1)
    return np.sum(i * x**2)
def alpine(x): return np.sum(np.abs(x*np.sin(x) + 0.1*x))
def griewank(x):
    i = np.arange(1, x.size+1)
    return np.sum(x**2)/4000.0 - np.prod(np.cos(x/np.sqrt(i))) + 1.0
def rastrigin(x): return 10.0*x.size + np.sum(x**2 - 10.0*np.cos(2*np.pi*x))
def ackley(x):
    d = x.size
    a, b, c = 20.0, 0.2, 2*np.pi
    s1 = np.sum(x**2)
    s2 = np.sum(np.cos(c*x))
    return (-a*np.exp(-b*np.sqrt(s1/d)) - np.exp(s2/d) + a + np.e)
def levy(x):
    w = 1 + (x - 1)/4
    term1 = np.sin(np.pi*w[0])**2
    term3 = (w[-1]-1)**2 * (1 + np.sin(2*np.pi*w[-1])**2)
    term2 = np.sum((w[:-1]-1)**2 * (1 + 10*np.sin(np.pi*w[:-1] + 1)**2))
    return term1 + term2 + term3
def salomon(x):
    r = np.sqrt(np.sum(x**2))
    return 1 - np.cos(2*np.pi*r) + 0.1*r
def schaffer(x):
    total = 0.0
    for i in range(x.size - 1):
        xi, xj = x[i], x[i+1]
        num = np.sin(np.sqrt(xi**2 + xj**2))**2 - 0.5
        den = (1 + 0.001*(xi**2 + xj**2))**2
        total += 0.5 + num/den
    return total

benchmark_funcs = [
    ("Sphere", sphere),
    ("Step", step),
    ("Schwefel 2.21", schwefel_221),
    ("Schwefel 2.22", schwefel_222),
    ("Rosenbrock", rosenbrock),
    ("BentCigar", bent_cigar),
    ("Sumsquares2", sumsquares2),
    ("Alpine", alpine),
    ("Griewank", griewank),
    ("Rastrigin", rastrigin),
    ("Ackley", ackley),
    ("Levy", levy),
    ("Salomon", salomon),
    ("Schaffer", schaffer),
]

# ============================================================
# 2) Modified HEO + (1+1)-ES WITHOUT phases
#    - r2 removed
#    - vibration anneals to 0 (still always present, but vanishes)
#    - escape multiplier bounded: m = 1 + log1p(c) (still increases with c)
#    - ES sigma dimension-aware
#    - random skip uses symmetric U(-b,b) (not U(0,b)) to avoid bias to large |x|
# ============================================================
class HEO_ES_NoPhases:
    def __init__(
        self,
        func,
        dim,
        bound=100.0,
        swarm_size=100,
        max_iters=1000,
        c_max=30,
        R=1.0,
        # vibration
        vib_sigma0=0.10,      # base coefficient (multiplied by anneal)
        vib_schedule="exp",   # "exp" or "linear"
        # ES
        es_window=20,
        es_c=0.85,
        es_sigma0_scale=0.10, # sigma0 = scale * (range)/sqrt(dim)
        # escape multiplier cap (optional safety)
        m_cap=10.0,           # cap on multiplier after log growth (keeps "escape" but stops blow-ups)
        seed=None
    ):
        self.func = func
        self.dim = int(dim)
        self.bound = float(bound)
        self.N = int(swarm_size)
        self.T = int(max_iters)
        self.c_max = int(c_max)
        self.R = float(R)

        self.vib_sigma0 = float(vib_sigma0)
        self.vib_schedule = vib_schedule

        self.es_window = int(es_window)
        self.es_c = float(es_c)
        self.es_sigma0_scale = float(es_sigma0_scale)

        self.m_cap = float(m_cap)

        self.lower = -self.bound * np.ones(self.dim)
        self.upper =  self.bound * np.ones(self.dim)
        self.rng = np.random.default_rng(seed)

        # range width
        self.range_width = (self.upper - self.lower)[0]

    def _anneal(self, t):
        if self.vib_schedule == "linear":
            return max(0.0, 1.0 - (t / max(1, self.T - 1)))
        return float(np.exp(-t / max(1, self.T)))

    def run(self, verbose=False):
        # init swarm
        X = self.rng.uniform(self.lower, self.upper, size=(self.N, self.dim))
        f = np.array([self.func(x) for x in X], dtype=float)

        pbest = X.copy()
        pbest_f = f.copy()

        gidx = int(np.argmin(f))
        gbest = X[gidx].copy()
        gbest_f = float(f[gidx])

        # per-particle escape counters
        c = np.zeros(self.N, dtype=int)

        # ES sigma dimension-aware
        sigma0 = self.es_sigma0_scale * (self.range_width / np.sqrt(self.dim))
        sigma = np.full(self.N, sigma0, dtype=float)

        succ = np.zeros(self.N, dtype=int)
        trials = np.zeros(self.N, dtype=int)

        for t in range(self.T):
            anneal = self._anneal(t)

            # swarm std for vibration scale (adaptive but vanishes)
            pop_std = np.std(X, axis=0)
            pop_std = np.where(pop_std == 0.0, 1e-12, pop_std)

            for i in range(self.N):
                x = X[i]

                # bounded escape multiplier: grows with c but not explosively
                m = 1.0 + np.log1p(c[i])
                if m > self.m_cap:
                    m = self.m_cap

                # --- HEO update (r2 removed) ---
                r1 = self.rng.uniform(1.0 - self.R, 1.0 + self.R, size=self.dim)
                r3 = self.rng.uniform(0.0, 1.0, size=self.dim)

                term = x * m * r1
                v_g  = (gbest - term) * r3
                v_l  = (pbest[i] - term) * (1.0 - r3)

                x_new = x + v_g + v_l

                # --- annealed vibration (goes to 0) ---
                x_new = x_new + (anneal * self.vib_sigma0) * (pop_std * self.rng.normal(0.0, 1.0, size=self.dim))

                x_new = np.clip(x_new, self.lower, self.upper)

                # --- (1+1)-ES local search ---
                x_mut = x_new + sigma[i] * self.rng.normal(0.0, 1.0, size=self.dim)
                x_mut = np.clip(x_mut, self.lower, self.upper)
                f_mut = float(self.func(x_mut))

                trials[i] += 1
                if f_mut < f[i]:
                    X[i] = x_mut
                    f[i] = f_mut
                    succ[i] += 1
                    c[i] = 0
                else:
                    c[i] += 1

                # update personal best
                if f[i] < pbest_f[i]:
                    pbest[i] = X[i].copy()
                    pbest_f[i] = f[i]

                # update global best
                if f[i] < gbest_f:
                    gbest = X[i].copy()
                    gbest_f = float(f[i])

                # --- 1/5 success rule every window ---
                if trials[i] >= self.es_window:
                    rate = succ[i] / trials[i]
                    if rate > 0.2:
                        sigma[i] /= self.es_c
                    else:
                        sigma[i] *= self.es_c
                    sigma[i] = float(np.clip(sigma[i], 1e-14, self.bound))
                    succ[i] = 0
                    trials[i] = 0

                # --- random skip if stagnating (escape, but unbiased) ---
                if c[i] > self.c_max:
                    # unbiased skip: symmetric range avoids pushing |x| large systematically
                    X[i] = self.rng.uniform(-self.bound, self.bound, size=self.dim)
                    f[i] = float(self.func(X[i]))
                    c[i] = 0
                    # keep sigma (do not reset), to preserve local-search "memory"

            if verbose and (t % max(1, self.T // 10) == 0):
                print(f"iter={t:4d}, gbest={gbest_f:.6e}")

        return gbest, gbest_f

# ============================================================
# 3) Run benchmark protocol
# ============================================================
DIM = 30
BOUND = 100.0
SWARM_SIZE = 100
MAX_ITERS = 1000
RUNS = 30  # if slow: 10 first, then 30

# core params
R = 1.0
C_MAX = 30

# vibration (convergent)
VIB_SIGMA0 = 0.10
VIB_SCHEDULE = "exp"

# ES params
ES_WINDOW = 20
ES_C = 0.85
ES_SIGMA0_SCALE = 0.10  # sigma0 = 0.10 * (range)/sqrt(dim)  -> ~3.65 for 30D, bound=100

# bounded escape multiplier
M_CAP = 10.0  # still escapes strongly, but prevents ballistic term growth

results = []

for fname, func in benchmark_funcs:
    best_vals = []
    times = []
    print(f"Running HEO+ES (no phases, bounded escape multiplier) on {fname} ...")

    for run in range(RUNS):
        opt = HEO_ES_NoPhases(
            func=func,
            dim=DIM,
            bound=BOUND,
            swarm_size=SWARM_SIZE,
            max_iters=MAX_ITERS,
            c_max=C_MAX,
            R=R,
            vib_sigma0=VIB_SIGMA0,
            vib_schedule=VIB_SCHEDULE,
            es_window=ES_WINDOW,
            es_c=ES_C,
            es_sigma0_scale=ES_SIGMA0_SCALE,
            m_cap=M_CAP,
            seed=run + 1234
        )

        t0 = time.time()
        _, best_f = opt.run(verbose=False)
        t1 = time.time()

        best_vals.append(best_f)
        times.append(t1 - t0)

    best_vals = np.array(best_vals, dtype=float)
    times = np.array(times, dtype=float)

    results.append({
        "Function": fname,
        "HEO+ES_mean_cost": float(best_vals.mean()),
        "HEO+ES_std_cost": float(best_vals.std()),
        "time_s_per_1000iters": float(times.mean()),
    })

df = pd.DataFrame(results)
display(df)

print("\nNotes / knobs (no phases used):")
print("- If Schwefel 2.22 still explodes, reduce M_CAP (e.g., 6.0) or VIB_SIGMA0 (e.g., 0.05).")
print("- For tighter convergence on Sphere/Step/Rastrigin, reduce VIB_SIGMA0 or ES_SIGMA0_SCALE a bit.")


Running HEO+ES (no phases, bounded escape multiplier) on Sphere ...
Running HEO+ES (no phases, bounded escape multiplier) on Step ...
Running HEO+ES (no phases, bounded escape multiplier) on Schwefel 2.21 ...
Running HEO+ES (no phases, bounded escape multiplier) on Schwefel 2.22 ...
Running HEO+ES (no phases, bounded escape multiplier) on Rosenbrock ...
Running HEO+ES (no phases, bounded escape multiplier) on BentCigar ...
Running HEO+ES (no phases, bounded escape multiplier) on Sumsquares2 ...
Running HEO+ES (no phases, bounded escape multiplier) on Alpine ...
Running HEO+ES (no phases, bounded escape multiplier) on Griewank ...
Running HEO+ES (no phases, bounded escape multiplier) on Rastrigin ...
Running HEO+ES (no phases, bounded escape multiplier) on Ackley ...
Running HEO+ES (no phases, bounded escape multiplier) on Levy ...
Running HEO+ES (no phases, bounded escape multiplier) on Salomon ...
Running HEO+ES (no phases, bounded escape multiplier) on Schaffer ...


,Function,HEO+ES_mean_cost,HEO+ES_std_cost,time_s_per_1000iters
0,Sphere,4.231990e+01,1.093100e+01,4.275032
1,Step,4.553333e+01,9.831017e+00,4.183220
2,Schwefel 2.21,1.544590e+00,2.912615e-01,3.931585
3,Schwefel 2.22,1.487381e+01,1.578537e+00,4.287704
4,Rosenbrock,1.078093e+04,4.342986e+03,4.712986
5,BentCigar,3.791177e+07,9.010660e+06,4.142658
6,Sumsquares2,4.913310e+02,1.228133e+02,4.360677
7,Alpine,1.318615e+01,4.747663e+00,4.333578
8,Griewank,9.226221e-01,4.595656e-02,5.049263
9,Rastrigin,2.973285e+02,2.092879e+01,4.446256



Notes / knobs (no phases used):
- If Schwefel 2.22 still explodes, reduce M_CAP (e.g., 6.0) or VIB_SIGMA0 (e.g., 0.05).
- For tighter convergence on Sphere/Step/Rastrigin, reduce VIB_SIGMA0 or ES_SIGMA0_SCALE a bit.


In [ ]:
import numpy as np
import pandas as pd
import time

# ============================================================
# 1) Benchmark functions
# ============================================================
def sphere(x): return np.sum(x**2)
def step(x): return np.sum(np.floor(x + 0.5)**2)
def schwefel_221(x): return np.max(np.abs(x))
def schwefel_222(x):
    a = np.abs(x)
    return np.sum(a) + np.prod(a)
def rosenbrock(x): return np.sum(100.0*(x[1:] - x[:-1]**2)**2 + (1-x[:-1])**2)
def bent_cigar(x): return x[0]**2 + 1e6*np.sum(x[1:]**2)
def sumsquares2(x):
    i = np.arange(1, x.size+1)
    return np.sum(i * x**2)
def alpine(x): return np.sum(np.abs(x*np.sin(x) + 0.1*x))
def griewank(x):
    i = np.arange(1, x.size+1)
    return np.sum(x**2)/4000.0 - np.prod(np.cos(x/np.sqrt(i))) + 1.0
def rastrigin(x): return 10.0*x.size + np.sum(x**2 - 10.0*np.cos(2*np.pi*x))
def ackley(x):
    d = x.size
    a, b, c = 20.0, 0.2, 2*np.pi
    s1 = np.sum(x**2)
    s2 = np.sum(np.cos(c*x))
    return (-a*np.exp(-b*np.sqrt(s1/d)) - np.exp(s2/d) + a + np.e)
def levy(x):
    w = 1 + (x - 1)/4
    term1 = np.sin(np.pi*w[0])**2
    term3 = (w[-1]-1)**2 * (1 + np.sin(2*np.pi*w[-1])**2)
    term2 = np.sum((w[:-1]-1)**2 * (1 + 10*np.sin(np.pi*w[:-1] + 1)**2))
    return term1 + term2 + term3
def salomon(x):
    r = np.sqrt(np.sum(x**2))
    return 1 - np.cos(2*np.pi*r) + 0.1*r
def schaffer(x):
    total = 0.0
    for i in range(x.size - 1):
        xi, xj = x[i], x[i+1]
        num = np.sin(np.sqrt(xi**2 + xj**2))**2 - 0.5
        den = (1 + 0.001*(xi**2 + xj**2))**2
        total += 0.5 + num/den
    return total

benchmark_funcs = [
    ("Sphere", sphere),
    ("Step", step),
    ("Schwefel 2.21", schwefel_221),
    ("Schwefel 2.22", schwefel_222),
    ("Rosenbrock", rosenbrock),
    ("BentCigar", bent_cigar),
    ("Sumsquares2", sumsquares2),
    ("Alpine", alpine),
    ("Griewank", griewank),
    ("Rastrigin", rastrigin),
    ("Ackley", ackley),
    ("Levy", levy),
    ("Salomon", salomon),
    ("Schaffer", schaffer),
]

# ============================================================
# 2) Modified HEO + (1+1)-ES WITHOUT phases
#    - r2 removed
#    - vibration anneals to 0 (still always present, but vanishes)
#    - escape multiplier bounded: m = 1 + log1p(c) (still increases with c)
#    - ES sigma dimension-aware
#    - random skip uses symmetric U(-b,b) (not U(0,b)) to avoid bias to large |x|
# ============================================================
class HEO_ES_NoPhases:
    def __init__(
        self,
        func,
        dim,
        bound=100.0,
        swarm_size=100,
        max_iters=1000,
        c_max=30,
        R=1.0,
        # vibration
        vib_sigma0=0.10,      # base coefficient (multiplied by anneal)
        vib_schedule="exp",   # "exp" or "linear"
        # ES
        es_window=20,
        es_c=0.85,
        es_sigma0_scale=0.10, # sigma0 = scale * (range)/sqrt(dim)
        # escape multiplier cap (optional safety)
        m_cap=10.0,           # cap on multiplier after log growth (keeps "escape" but stops blow-ups)
        seed=None
    ):
        self.func = func
        self.dim = int(dim)
        self.bound = float(bound)
        self.N = int(swarm_size)
        self.T = int(max_iters)
        self.c_max = int(c_max)
        self.R = float(R)

        self.vib_sigma0 = float(vib_sigma0)
        self.vib_schedule = vib_schedule

        self.es_window = int(es_window)
        self.es_c = float(es_c)
        self.es_sigma0_scale = float(es_sigma0_scale)

        self.m_cap = float(m_cap)

        self.lower = -self.bound * np.ones(self.dim)
        self.upper =  self.bound * np.ones(self.dim)
        self.rng = np.random.default_rng(seed)

        # range width
        self.range_width = (self.upper - self.lower)[0]

    def _anneal(self, t):
        if self.vib_schedule == "linear":
            return max(0.0, 1.0 - (t / max(1, self.T - 1)))
        return float(np.exp(-t / max(1, self.T)))

    def run(self, verbose=False):
        # init swarm
        X = self.rng.uniform(self.lower, self.upper, size=(self.N, self.dim))
        f = np.array([self.func(x) for x in X], dtype=float)

        pbest = X.copy()
        pbest_f = f.copy()

        gidx = int(np.argmin(f))
        gbest = X[gidx].copy()
        gbest_f = float(f[gidx])

        # per-particle escape counters
        c = np.zeros(self.N, dtype=int)

        # ES sigma dimension-aware
        sigma0 = self.es_sigma0_scale * (self.range_width / np.sqrt(self.dim))
        sigma = np.full(self.N, sigma0, dtype=float)

        succ = np.zeros(self.N, dtype=int)
        trials = np.zeros(self.N, dtype=int)

        for t in range(self.T):
            anneal = self._anneal(t)

            # swarm std for vibration scale (adaptive but vanishes)
            pop_std = np.std(X, axis=0)
            pop_std = np.where(pop_std == 0.0, 1e-12, pop_std)

            for i in range(self.N):
                x = X[i]

                # bounded escape multiplier: grows with c but not explosively
                m = 1.0 + np.log1p(c[i])
                if m > self.m_cap:
                    m = self.m_cap

                # --- HEO update (r2 removed) ---
                r1 = self.rng.uniform(1.0 - self.R, 1.0 + self.R, size=self.dim)
                r3 = self.rng.uniform(0.0, 1.0, size=self.dim)

                term = x * m * r1
                v_g  = (gbest - term) * r3
                v_l  = (pbest[i] - term) * (1.0 - r3)

                x_new = x + v_g + v_l

                # --- annealed vibration (goes to 0) ---
                x_new = x_new + (anneal * self.vib_sigma0) * (pop_std * self.rng.normal(0.0, 1.0, size=self.dim))

                x_new = np.clip(x_new, self.lower, self.upper)

                # --- (1+1)-ES local search ---
                x_mut = x_new + sigma[i] * self.rng.normal(0.0, 1.0, size=self.dim)
                x_mut = np.clip(x_mut, self.lower, self.upper)
                f_mut = float(self.func(x_mut))

                trials[i] += 1
                if f_mut < f[i]:
                    X[i] = x_mut
                    f[i] = f_mut
                    succ[i] += 1
                    c[i] = 0
                else:
                    c[i] += 1

                # update personal best
                if f[i] < pbest_f[i]:
                    pbest[i] = X[i].copy()
                    pbest_f[i] = f[i]

                # update global best
                if f[i] < gbest_f:
                    gbest = X[i].copy()
                    gbest_f = float(f[i])

                # --- 1/5 success rule every window ---
                if trials[i] >= self.es_window:
                    rate = succ[i] / trials[i]
                    if rate > 0.2:
                        sigma[i] /= self.es_c
                    else:
                        sigma[i] *= self.es_c
                    sigma[i] = float(np.clip(sigma[i], 1e-14, self.bound))
                    succ[i] = 0
                    trials[i] = 0

                # --- random skip if stagnating (escape, but unbiased) ---
                if c[i] > self.c_max:
                    # unbiased skip: symmetric range avoids pushing |x| large systematically
                    X[i] = self.rng.uniform(-self.bound, self.bound, size=self.dim)
                    f[i] = float(self.func(X[i]))
                    c[i] = 0
                    # keep sigma (do not reset), to preserve local-search "memory"

            if verbose and (t % max(1, self.T // 10) == 0):
                print(f"iter={t:4d}, gbest={gbest_f:.6e}")

        return gbest, gbest_f

# ============================================================
# 3) Run benchmark protocol
# ============================================================
DIM = 30
BOUND = 100.0
SWARM_SIZE = 100
MAX_ITERS = 1000
RUNS = 30  # if slow: 10 first, then 30

# core params
R = 1.0
C_MAX = 30

# vibration (convergent)
VIB_SIGMA0 = 0.10
VIB_SCHEDULE = "exp"

# ES params
ES_WINDOW = 20
ES_C = 0.85
ES_SIGMA0_SCALE = 0.10  # sigma0 = 0.10 * (range)/sqrt(dim)  -> ~3.65 for 30D, bound=100

# bounded escape multiplier
M_CAP = 10.0  # still escapes strongly, but prevents ballistic term growth

results = []

for fname, func in benchmark_funcs:
    best_vals = []
    times = []
    print(f"Running HEO+ES (no phases, bounded escape multiplier) on {fname} ...")

    for run in range(RUNS):
        opt = HEO_ES_NoPhases(
            func=func,
            dim=DIM,
            bound=BOUND,
            swarm_size=SWARM_SIZE,
            max_iters=MAX_ITERS,
            c_max=C_MAX,
            R=R,
            vib_sigma0=VIB_SIGMA0,
            vib_schedule=VIB_SCHEDULE,
            es_window=ES_WINDOW,
            es_c=ES_C,
            es_sigma0_scale=ES_SIGMA0_SCALE,
            m_cap=M_CAP,
            seed=run + 1234
        )

        t0 = time.time()
        _, best_f = opt.run(verbose=False)
        t1 = time.time()

        best_vals.append(best_f)
        times.append(t1 - t0)

    best_vals = np.array(best_vals, dtype=float)
    times = np.array(times, dtype=float)

    results.append({
        "Function": fname,
        "HEO+ES_mean_cost": float(best_vals.mean()),
        "HEO+ES_std_cost": float(best_vals.std()),
        "time_s_per_1000iters": float(times.mean()),
    })

df = pd.DataFrame(results)
display(df)

print("\nNotes / knobs (no phases used):")
print("- If Schwefel 2.22 still explodes, reduce M_CAP (e.g., 6.0) or VIB_SIGMA0 (e.g., 0.05).")
print("- For tighter convergence on Sphere/Step/Rastrigin, reduce VIB_SIGMA0 or ES_SIGMA0_SCALE a bit.")


Running HEO+ES (no phases, bounded escape multiplier) on Sphere ...
Running HEO+ES (no phases, bounded escape multiplier) on Step ...
Running HEO+ES (no phases, bounded escape multiplier) on Schwefel 2.21 ...
Running HEO+ES (no phases, bounded escape multiplier) on Schwefel 2.22 ...
Running HEO+ES (no phases, bounded escape multiplier) on Rosenbrock ...
Running HEO+ES (no phases, bounded escape multiplier) on BentCigar ...
Running HEO+ES (no phases, bounded escape multiplier) on Sumsquares2 ...
Running HEO+ES (no phases, bounded escape multiplier) on Alpine ...
Running HEO+ES (no phases, bounded escape multiplier) on Griewank ...
Running HEO+ES (no phases, bounded escape multiplier) on Rastrigin ...
Running HEO+ES (no phases, bounded escape multiplier) on Ackley ...
Running HEO+ES (no phases, bounded escape multiplier) on Levy ...
Running HEO+ES (no phases, bounded escape multiplier) on Salomon ...
Running HEO+ES (no phases, bounded escape multiplier) on Schaffer ...


,Function,HEO+ES_mean_cost,HEO+ES_std_cost,time_s_per_1000iters
0,Sphere,4.231990e+01,1.093100e+01,4.261514
1,Step,4.553333e+01,9.831017e+00,4.290006
2,Schwefel 2.21,1.544590e+00,2.912615e-01,3.907142
3,Schwefel 2.22,1.487381e+01,1.578537e+00,4.274243
4,Rosenbrock,1.078093e+04,4.342986e+03,4.674033
5,BentCigar,3.791177e+07,9.010660e+06,4.128279
6,Sumsquares2,4.913310e+02,1.228133e+02,4.315879
7,Alpine,1.318615e+01,4.747663e+00,4.336157
8,Griewank,9.226221e-01,4.595656e-02,5.016316
9,Rastrigin,2.973285e+02,2.092879e+01,4.473674



Notes / knobs (no phases used):
- If Schwefel 2.22 still explodes, reduce M_CAP (e.g., 6.0) or VIB_SIGMA0 (e.g., 0.05).
- For tighter convergence on Sphere/Step/Rastrigin, reduce VIB_SIGMA0 or ES_SIGMA0_SCALE a bit.


In [ ]:
import numpy as np
import pandas as pd
import time

# ============================================================
# 1) Benchmark functions
# ============================================================
def sphere(x): return np.sum(x**2)
def step(x): return np.sum(np.floor(x + 0.5)**2)
def schwefel_221(x): return np.max(np.abs(x))
def schwefel_222(x):
    a = np.abs(x)
    return np.sum(a) + np.prod(a)
def rosenbrock(x): return np.sum(100.0*(x[1:] - x[:-1]**2)**2 + (1-x[:-1])**2)
def bent_cigar(x): return x[0]**2 + 1e6*np.sum(x[1:]**2)
def sumsquares2(x):
    i = np.arange(1, x.size+1)
    return np.sum(i * x**2)
def alpine(x): return np.sum(np.abs(x*np.sin(x) + 0.1*x))
def griewank(x):
    i = np.arange(1, x.size+1)
    return np.sum(x**2)/4000.0 - np.prod(np.cos(x/np.sqrt(i))) + 1.0
def rastrigin(x): return 10.0*x.size + np.sum(x**2 - 10.0*np.cos(2*np.pi*x))
def ackley(x):
    d = x.size
    a, b, c = 20.0, 0.2, 2*np.pi
    s1 = np.sum(x**2)
    s2 = np.sum(np.cos(c*x))
    return (-a*np.exp(-b*np.sqrt(s1/d)) - np.exp(s2/d) + a + np.e)
def levy(x):
    w = 1 + (x - 1)/4
    term1 = np.sin(np.pi*w[0])**2
    term3 = (w[-1]-1)**2 * (1 + np.sin(2*np.pi*w[-1])**2)
    term2 = np.sum((w[:-1]-1)**2 * (1 + 10*np.sin(np.pi*w[:-1] + 1)**2))
    return term1 + term2 + term3
def salomon(x):
    r = np.sqrt(np.sum(x**2))
    return 1 - np.cos(2*np.pi*r) + 0.1*r
def schaffer(x):
    total = 0.0
    for i in range(x.size - 1):
        xi, xj = x[i], x[i+1]
        num = np.sin(np.sqrt(xi**2 + xj**2))**2 - 0.5
        den = (1 + 0.001*(xi**2 + xj**2))**2
        total += 0.5 + num/den
    return total

benchmark_funcs = [
    ("Sphere", sphere),
    ("Step", step),
    ("Schwefel 2.21", schwefel_221),
    ("Schwefel 2.22", schwefel_222),
    ("Rosenbrock", rosenbrock),
    ("BentCigar", bent_cigar),
    ("Sumsquares2", sumsquares2),
    ("Alpine", alpine),
    ("Griewank", griewank),
    ("Rastrigin", rastrigin),
    ("Ackley", ackley),
    ("Levy", levy),
    ("Salomon", salomon),
    ("Schaffer", schaffer),
]

# ============================================================
# 2) HEO+ES v3: escape probability (no phases)
# ============================================================
class HEO_ES_v3:
    def __init__(
        self,
        func,
        dim,
        bound=100.0,
        swarm_size=100,
        max_iters=1000,
        c_max=30,
        R=1.0,
        # vibration (vanishes)
        vib_sigma0=0.10,
        vib_schedule="exp",   # "exp" or "linear"
        # ES (1/5 rule)
        es_window=20,
        es_c=0.85,
        es_sigma0_scale=0.10, # sigma0 = scale*(range)/sqrt(dim)
        # escape multiplier
        m_cap=10.0,           # cap on 1+log1p(c)
        # escape probability
        p_min=0.05,           # minimum escape chance even if not stuck (keeps "escape always possible")
        seed=None
    ):
        self.func = func
        self.dim = int(dim)
        self.bound = float(bound)
        self.N = int(swarm_size)
        self.T = int(max_iters)

        self.c_max = int(c_max)
        self.R = float(R)

        self.vib_sigma0 = float(vib_sigma0)
        self.vib_schedule = vib_schedule

        self.es_window = int(es_window)
        self.es_c = float(es_c)
        self.es_sigma0_scale = float(es_sigma0_scale)

        self.m_cap = float(m_cap)
        self.p_min = float(p_min)

        self.lower = -self.bound * np.ones(self.dim)
        self.upper =  self.bound * np.ones(self.dim)
        self.rng = np.random.default_rng(seed)

        self.range_width = (self.upper - self.lower)[0]

    def _anneal(self, t):
        if self.vib_schedule == "linear":
            return max(0.0, 1.0 - (t / max(1, self.T - 1)))
        return float(np.exp(-t / max(1, self.T)))

    def run(self, verbose=False):
        # init
        X = self.rng.uniform(self.lower, self.upper, size=(self.N, self.dim))
        f = np.array([self.func(x) for x in X], dtype=float)

        pbest = X.copy()
        pbest_f = f.copy()

        gidx = int(np.argmin(f))
        gbest = X[gidx].copy()
        gbest_f = float(f[gidx])

        c = np.zeros(self.N, dtype=int)

        # ES sigma dimension-aware
        sigma0 = self.es_sigma0_scale * (self.range_width / np.sqrt(self.dim))
        sigma = np.full(self.N, sigma0, dtype=float)
        succ = np.zeros(self.N, dtype=int)
        trials = np.zeros(self.N, dtype=int)

        for t in range(self.T):
            anneal = self._anneal(t)
            pop_std = np.std(X, axis=0)
            pop_std = np.where(pop_std == 0.0, 1e-12, pop_std)

            for i in range(self.N):
                x = X[i]

                # escape probability increases with stagnation
                p_escape = max(self.p_min, min(1.0, c[i] / max(1, self.c_max)))

                # bounded escape multiplier (still grows with c)
                m = 1.0 + np.log1p(c[i])
                if m > self.m_cap:
                    m = self.m_cap

                r3 = self.rng.uniform(0.0, 1.0, size=self.dim)

                if self.rng.random() < p_escape:
                    # --- ESCAPE update (uses term) ---
                    r1 = self.rng.uniform(1.0 - self.R, 1.0 + self.R, size=self.dim)
                    term = x * m * r1
                else:
                    # --- NON-ESCAPE update (pure attraction) ---
                    # keeps exploitation stable near optimum
                    term = 0.0

                v_g = (gbest - term) * r3
                v_l = (pbest[i] - term) * (1.0 - r3)

                x_new = x + v_g + v_l

                # vibration that vanishes (still present always, but to 0)
                x_new = x_new + (anneal * self.vib_sigma0) * (pop_std * self.rng.normal(0.0, 1.0, size=self.dim))

                x_new = np.clip(x_new, self.lower, self.upper)

                # (1+1)-ES
                x_mut = x_new + sigma[i] * self.rng.normal(0.0, 1.0, size=self.dim)
                x_mut = np.clip(x_mut, self.lower, self.upper)
                f_mut = float(self.func(x_mut))

                trials[i] += 1
                if f_mut < f[i]:
                    X[i] = x_mut
                    f[i] = f_mut
                    succ[i] += 1
                    c[i] = 0
                else:
                    c[i] += 1

                # update pbest/gbest
                if f[i] < pbest_f[i]:
                    pbest[i] = X[i].copy()
                    pbest_f[i] = f[i]
                if f[i] < gbest_f:
                    gbest = X[i].copy()
                    gbest_f = float(f[i])

                # 1/5 success rule
                if trials[i] >= self.es_window:
                    rate = succ[i] / trials[i]
                    if rate > 0.2:
                        sigma[i] /= self.es_c
                    else:
                        sigma[i] *= self.es_c
                    sigma[i] = float(np.clip(sigma[i], 1e-14, self.bound))
                    succ[i] = 0
                    trials[i] = 0

                # random skip (escape) if too stagnant
                if c[i] > self.c_max:
                    X[i] = self.rng.uniform(-self.bound, self.bound, size=self.dim)
                    f[i] = float(self.func(X[i]))
                    c[i] = 0

            if verbose and (t % max(1, self.T // 10) == 0):
                print(f"iter={t:4d}, gbest={gbest_f:.6e}")

        return gbest, gbest_f

# ============================================================
# 3) Run benchmark protocol
# ============================================================
DIM = 30
BOUND = 100.0
SWARM_SIZE = 100
MAX_ITERS = 1000
RUNS = 30  # if slow: 10 then 30

# params (start from your previous good settings)
R = 1.0
C_MAX = 30
M_CAP = 10.0

VIB_SIGMA0 = 0.10
VIB_SCHEDULE = "exp"

ES_WINDOW = 20
ES_C = 0.85
ES_SIGMA0_SCALE = 0.10

# escape prob: ensure escape always possible but mostly when stagnant
P_MIN = 0.05

results = []

for fname, func in benchmark_funcs:
    best_vals = []
    times = []
    print(f"Running HEO+ES v3 (escape probability) on {fname} ...")

    for run in range(RUNS):
        opt = HEO_ES_v3(
            func=func,
            dim=DIM,
            bound=BOUND,
            swarm_size=SWARM_SIZE,
            max_iters=MAX_ITERS,
            c_max=C_MAX,
            R=R,
            vib_sigma0=VIB_SIGMA0,
            vib_schedule=VIB_SCHEDULE,
            es_window=ES_WINDOW,
            es_c=ES_C,
            es_sigma0_scale=ES_SIGMA0_SCALE,
            m_cap=M_CAP,
            p_min=P_MIN,
            seed=run + 1234
        )

        t0 = time.time()
        _, best_f = opt.run(verbose=False)
        t1 = time.time()

        best_vals.append(best_f)
        times.append(t1 - t0)

    best_vals = np.array(best_vals, dtype=float)
    times = np.array(times, dtype=float)

    results.append({
        "Function": fname,
        "HEO+ESv3_mean_cost": float(best_vals.mean()),
        "HEO+ESv3_std_cost": float(best_vals.std()),
        "time_s_per_1000iters": float(times.mean()),
    })

df = pd.DataFrame(results)
display(df)

print("\nTuning hints (still no phases):")
print("- If Sphere/Step not near 0: lower P_MIN to 0.01 and/or VIB_SIGMA0 to 0.05.")
print("- If Schwefel 2.22 worsens: lower M_CAP to 6.0.")
print("- If Rosenbrock/BentCigar still large: try ES_SIGMA0_SCALE=0.05.")


Running HEO+ES v3 (escape probability) on Sphere ...
Running HEO+ES v3 (escape probability) on Step ...
Running HEO+ES v3 (escape probability) on Schwefel 2.21 ...
Running HEO+ES v3 (escape probability) on Schwefel 2.22 ...
Running HEO+ES v3 (escape probability) on Rosenbrock ...
Running HEO+ES v3 (escape probability) on BentCigar ...
Running HEO+ES v3 (escape probability) on Sumsquares2 ...
Running HEO+ES v3 (escape probability) on Alpine ...
Running HEO+ES v3 (escape probability) on Griewank ...
Running HEO+ES v3 (escape probability) on Rastrigin ...
Running HEO+ES v3 (escape probability) on Ackley ...
Running HEO+ES v3 (escape probability) on Levy ...
Running HEO+ES v3 (escape probability) on Salomon ...
Running HEO+ES v3 (escape probability) on Schaffer ...


,Function,HEO+ESv3_mean_cost,HEO+ESv3_std_cost,time_s_per_1000iters
0,Sphere,2.362607e+04,6.509467e+03,4.298445
1,Step,2.521903e+04,4.819081e+03,4.271337
2,Schwefel 2.21,6.936354e+01,5.725672e+00,3.978329
3,Schwefel 2.22,3.958528e+33,1.975050e+34,4.273933
4,Rosenbrock,6.347645e+09,2.300211e+09,4.724712
5,BentCigar,2.270812e+10,4.611482e+09,4.147859
6,Sumsquares2,3.354447e+05,7.763277e+04,4.402051
7,Alpine,4.012363e+02,4.411079e+01,4.479902
8,Griewank,6.906517e+00,1.627367e+00,5.260881
9,Rastrigin,2.524745e+04,5.430040e+03,4.610440



Tuning hints (still no phases):
- If Sphere/Step not near 0: lower P_MIN to 0.01 and/or VIB_SIGMA0 to 0.05.
- If Schwefel 2.22 worsens: lower M_CAP to 6.0.
- If Rosenbrock/BentCigar still large: try ES_SIGMA0_SCALE=0.05.


In [ ]:
import numpy as np
import pandas as pd
import time

# ============================================================
# 1) Benchmark functions
# ============================================================
def sphere(x): return np.sum(x**2)
def step(x): return np.sum(np.floor(x + 0.5)**2)
def schwefel_221(x): return np.max(np.abs(x))
def schwefel_222(x):
    a = np.abs(x)
    return np.sum(a) + np.prod(a)
def rosenbrock(x): return np.sum(100.0*(x[1:] - x[:-1]**2)**2 + (1-x[:-1])**2)
def bent_cigar(x): return x[0]**2 + 1e6*np.sum(x[1:]**2)
def sumsquares2(x):
    i = np.arange(1, x.size+1)
    return np.sum(i * x**2)
def alpine(x): return np.sum(np.abs(x*np.sin(x) + 0.1*x))
def griewank(x):
    i = np.arange(1, x.size+1)
    return np.sum(x**2)/4000.0 - np.prod(np.cos(x/np.sqrt(i))) + 1.0
def rastrigin(x): return 10.0*x.size + np.sum(x**2 - 10.0*np.cos(2*np.pi*x))
def ackley(x):
    d = x.size
    a, b, c = 20.0, 0.2, 2*np.pi
    s1 = np.sum(x**2)
    s2 = np.sum(np.cos(c*x))
    return (-a*np.exp(-b*np.sqrt(s1/d)) - np.exp(s2/d) + a + np.e)
def levy(x):
    w = 1 + (x - 1)/4
    term1 = np.sin(np.pi*w[0])**2
    term3 = (w[-1]-1)**2 * (1 + np.sin(2*np.pi*w[-1])**2)
    term2 = np.sum((w[:-1]-1)**2 * (1 + 10*np.sin(np.pi*w[:-1] + 1)**2))
    return term1 + term2 + term3
def salomon(x):
    r = np.sqrt(np.sum(x**2))
    return 1 - np.cos(2*np.pi*r) + 0.1*r
def schaffer(x):
    total = 0.0
    for i in range(x.size - 1):
        xi, xj = x[i], x[i+1]
        num = np.sin(np.sqrt(xi**2 + xj**2))**2 - 0.5
        den = (1 + 0.001*(xi**2 + xj**2))**2
        total += 0.5 + num/den
    return total

benchmark_funcs = [
    ("Sphere", sphere),
    ("Step", step),
    ("Schwefel 2.21", schwefel_221),
    ("Schwefel 2.22", schwefel_222),
    ("Rosenbrock", rosenbrock),
    ("BentCigar", bent_cigar),
    ("Sumsquares2", sumsquares2),
    ("Alpine", alpine),
    ("Griewank", griewank),
    ("Rastrigin", rastrigin),
    ("Ackley", ackley),
    ("Levy", levy),
    ("Salomon", salomon),
    ("Schaffer", schaffer),
]

# ============================================================
# 2) HEO + CIBA-style adaptive mutation (no ES)
#    - r2 removed
#    - bounded escape multiplier
#    - annealed vibration -> 0
#    - mutation strength grows with (f_i - f_best)
# ============================================================
class HEO_CIBA:
    def __init__(
        self,
        func,
        dim,
        bound=100.0,
        swarm_size=100,
        max_iters=1000,
        c_max=30,
        R=1.0,
        # vibration
        vib_sigma0=0.10,
        vib_schedule="exp",   # "exp" or "linear"
        # bounded escape multiplier
        m_cap=10.0,
        # CIBA mutation params
        beta=5.0,            # max mutation scale
        k_aff=1.0,           # sensitivity of exp() to aff
        eps=1e-12,
        seed=None
    ):
        self.func = func
        self.dim = int(dim)
        self.bound = float(bound)
        self.N = int(swarm_size)
        self.T = int(max_iters)
        self.c_max = int(c_max)
        self.R = float(R)

        self.vib_sigma0 = float(vib_sigma0)
        self.vib_schedule = vib_schedule
        self.m_cap = float(m_cap)

        self.beta = float(beta)
        self.k_aff = float(k_aff)
        self.eps = float(eps)

        self.lower = -self.bound * np.ones(self.dim)
        self.upper =  self.bound * np.ones(self.dim)
        self.rng = np.random.default_rng(seed)

    def _anneal(self, t):
        if self.vib_schedule == "linear":
            return max(0.0, 1.0 - (t / max(1, self.T - 1)))
        return float(np.exp(-t / max(1, self.T)))

    def run(self, verbose=False):
        X = self.rng.uniform(self.lower, self.upper, size=(self.N, self.dim))
        f = np.array([self.func(x) for x in X], dtype=float)

        pbest = X.copy()
        pbest_f = f.copy()

        gidx = int(np.argmin(f))
        gbest = X[gidx].copy()
        gbest_f = float(f[gidx])

        c = np.zeros(self.N, dtype=int)

        for t in range(self.T):
            anneal = self._anneal(t)

            pop_std = np.std(X, axis=0)
            pop_std = np.where(pop_std == 0.0, 1e-12, pop_std)

            for i in range(self.N):
                x = X[i]

                # bounded escape multiplier
                m = 1.0 + np.log1p(c[i])
                if m > self.m_cap:
                    m = self.m_cap

                # HEO update without r2
                r1 = self.rng.uniform(1.0 - self.R, 1.0 + self.R, size=self.dim)
                r3 = self.rng.uniform(0.0, 1.0, size=self.dim)

                term = x * m * r1
                v_g  = (gbest - term) * r3
                v_l  = (pbest[i] - term) * (1.0 - r3)
                x_new = x + v_g + v_l

                # annealed vibration (vanishes)
                x_new = x_new + (anneal * self.vib_sigma0) * (pop_std * self.rng.normal(0.0, 1.0, size=self.dim))

                # ---- CIBA-style adaptive mutation ----
                # aff >= 0 (how worse than global best)
                aff = max(0.0, (f[i] - gbest_f) / (abs(gbest_f) + 1.0 + self.eps))

                # reverse-distance mutation factor: close => ~0, far => ~1
                mut_factor = 1.0 - np.exp(-self.k_aff * aff)

                # mutation vector
                r = self.rng.normal(0.0, 1.0, size=self.dim)
                x_mut = x_new + (mut_factor * self.beta) * r

                x_mut = np.clip(x_mut, self.lower, self.upper)
                f_mut = float(self.func(x_mut))

                # accept if improved current position
                if f_mut < f[i]:
                    X[i] = x_mut
                    f[i] = f_mut
                    c[i] = 0
                else:
                    c[i] += 1

                # update personal / global best
                if f[i] < pbest_f[i]:
                    pbest[i] = X[i].copy()
                    pbest_f[i] = f[i]
                if f[i] < gbest_f:
                    gbest = X[i].copy()
                    gbest_f = float(f[i])

                # random skip if stagnating too long (unbiased)
                if c[i] > self.c_max:
                    X[i] = self.rng.uniform(-self.bound, self.bound, size=self.dim)
                    f[i] = float(self.func(X[i]))
                    c[i] = 0

            if verbose and (t % max(1, self.T // 10) == 0):
                print(f"iter={t:4d}, gbest={gbest_f:.6e}")

        return gbest, gbest_f

# ============================================================
# 3) Run benchmark protocol
# ============================================================
DIM = 30
BOUND = 100.0
SWARM_SIZE = 100
MAX_ITERS = 1000
RUNS = 30  # if slow: 10 first

R = 1.0
C_MAX = 30
M_CAP = 10.0

VIB_SIGMA0 = 0.10
VIB_SCHEDULE = "exp"

# CIBA mutation knobs
BETA = 5.0      # try 2.0..10.0
K_AFF = 2.0     # try 1.0..5.0 (higher => more aggressive when far)
EPS = 1e-12

results = []

for fname, func in benchmark_funcs:
    best_vals = []
    times = []
    print(f"Running HEO + CIBA-mutation on {fname} ...")

    for run in range(RUNS):
        opt = HEO_CIBA(
            func=func,
            dim=DIM,
            bound=BOUND,
            swarm_size=SWARM_SIZE,
            max_iters=MAX_ITERS,
            c_max=C_MAX,
            R=R,
            vib_sigma0=VIB_SIGMA0,
            vib_schedule=VIB_SCHEDULE,
            m_cap=M_CAP,
            beta=BETA,
            k_aff=K_AFF,
            eps=EPS,
            seed=run + 1234
        )

        t0 = time.time()
        _, best_f = opt.run(verbose=False)
        t1 = time.time()

        best_vals.append(best_f)
        times.append(t1 - t0)

    best_vals = np.array(best_vals, dtype=float)
    times = np.array(times, dtype=float)

    results.append({
        "Function": fname,
        "HEO+CIBA_mean_cost": float(best_vals.mean()),
        "HEO+CIBA_std_cost": float(best_vals.std()),
        "time_s_per_1000iters": float(times.mean()),
    })

df = pd.DataFrame(results)
display(df)

print("\nTuning tips:")
print("- If Sphere/Step not improving: lower BETA (e.g., 2.0) and/or VIB_SIGMA0 (0.05).")
print("- If getting stuck: increase K_AFF (e.g., 3.0 or 4.0).")
print("- If Schwefel 2.22 grows: reduce M_CAP (e.g., 6.0) and reduce BETA.")


Running HEO + CIBA-mutation on Sphere ...
Running HEO + CIBA-mutation on Step ...
Running HEO + CIBA-mutation on Schwefel 2.21 ...
Running HEO + CIBA-mutation on Schwefel 2.22 ...
Running HEO + CIBA-mutation on Rosenbrock ...
Running HEO + CIBA-mutation on BentCigar ...
Running HEO + CIBA-mutation on Sumsquares2 ...
Running HEO + CIBA-mutation on Alpine ...
Running HEO + CIBA-mutation on Griewank ...
Running HEO + CIBA-mutation on Rastrigin ...
Running HEO + CIBA-mutation on Ackley ...
Running HEO + CIBA-mutation on Levy ...
Running HEO + CIBA-mutation on Salomon ...
Running HEO + CIBA-mutation on Schaffer ...


,Function,HEO+CIBA_mean_cost,HEO+CIBA_std_cost,time_s_per_1000iters
0,Sphere,2.962553e+02,1.097706e+02,4.179194
1,Step,2.968000e+02,1.183172e+02,4.093675
2,Schwefel 2.21,9.085322e+00,1.259904e+00,3.863196
3,Schwefel 2.22,8.705956e+12,3.009069e+13,4.201175
4,Rosenbrock,1.952875e+06,1.083492e+06,4.640664
5,BentCigar,2.684098e+08,1.286546e+08,4.028999
6,Sumsquares2,3.788166e+03,1.592140e+03,4.226182
7,Alpine,4.668086e+01,1.160212e+01,4.293756
8,Griewank,7.387177e-01,1.020711e-01,5.000691
9,Rastrigin,4.854299e+02,8.067782e+01,4.495626



Tuning tips:
- If Sphere/Step not improving: lower BETA (e.g., 2.0) and/or VIB_SIGMA0 (0.05).
- If getting stuck: increase K_AFF (e.g., 3.0 or 4.0).
- If Schwefel 2.22 grows: reduce M_CAP (e.g., 6.0) and reduce BETA.


In [ ]:
import numpy as np
import pandas as pd
import time

# =========================
# Benchmarks
# =========================
def sphere(x): return np.sum(x**2)
def step(x): return np.sum(np.floor(x + 0.5)**2)
def schwefel_221(x): return np.max(np.abs(x))
def schwefel_222(x):
    a = np.abs(x)
    return np.sum(a) + np.prod(a)
def rosenbrock(x): return np.sum(100.0*(x[1:] - x[:-1]**2)**2 + (1-x[:-1])**2)
def bent_cigar(x): return x[0]**2 + 1e6*np.sum(x[1:]**2)
def sumsquares2(x):
    i = np.arange(1, x.size+1)
    return np.sum(i * x**2)
def alpine(x): return np.sum(np.abs(x*np.sin(x) + 0.1*x))
def griewank(x):
    i = np.arange(1, x.size+1)
    return np.sum(x**2)/4000.0 - np.prod(np.cos(x/np.sqrt(i))) + 1.0
def rastrigin(x): return 10.0*x.size + np.sum(x**2 - 10.0*np.cos(2*np.pi*x))
def ackley(x):
    d = x.size
    a, b, c = 20.0, 0.2, 2*np.pi
    s1 = np.sum(x**2)
    s2 = np.sum(np.cos(c*x))
    return (-a*np.exp(-b*np.sqrt(s1/d)) - np.exp(s2/d) + a + np.e)
def levy(x):
    w = 1 + (x - 1)/4
    term1 = np.sin(np.pi*w[0])**2
    term3 = (w[-1]-1)**2 * (1 + np.sin(2*np.pi*w[-1])**2)
    term2 = np.sum((w[:-1]-1)**2 * (1 + 10*np.sin(np.pi*w[:-1] + 1)**2))
    return term1 + term2 + term3
def salomon(x):
    r = np.sqrt(np.sum(x**2))
    return 1 - np.cos(2*np.pi*r) + 0.1*r
def schaffer(x):
    total = 0.0
    for i in range(x.size - 1):
        xi, xj = x[i], x[i+1]
        num = np.sin(np.sqrt(xi**2 + xj**2))**2 - 0.5
        den = (1 + 0.001*(xi**2 + xj**2))**2
        total += 0.5 + num/den
    return total

benchmark_funcs = [
    ("Sphere", sphere),
    ("Step", step),
    ("Schwefel 2.21", schwefel_221),
    ("Schwefel 2.22", schwefel_222),
    ("Rosenbrock", rosenbrock),
    ("BentCigar", bent_cigar),
    ("Sumsquares2", sumsquares2),
    ("Alpine", alpine),
    ("Griewank", griewank),
    ("Rastrigin", rastrigin),
    ("Ackley", ackley),
    ("Levy", levy),
    ("Salomon", salomon),
    ("Schaffer", schaffer),
]

# =========================
# HEO + Directional CIBA
# =========================
class HEO_DirectionalCIBA:
    def __init__(
        self,
        func, dim,
        bound=100.0, swarm_size=100, max_iters=1000,
        c_max=30, R=1.0,
        vib_sigma0=0.10, vib_schedule="exp",
        m_cap=10.0,
        beta=5.0, k_aff=2.0, alpha_dir=0.7,   # <-- directional mutation
        eps=1e-12,
        seed=None
    ):
        self.func = func
        self.dim = dim
        self.bound = float(bound)
        self.N = swarm_size
        self.T = max_iters
        self.c_max = c_max
        self.R = float(R)

        self.vib_sigma0 = float(vib_sigma0)
        self.vib_schedule = vib_schedule
        self.m_cap = float(m_cap)

        self.beta = float(beta)
        self.k_aff = float(k_aff)
        self.alpha_dir = float(alpha_dir)
        self.eps = float(eps)

        self.lower = -self.bound*np.ones(dim)
        self.upper =  self.bound*np.ones(dim)
        self.rng = np.random.default_rng(seed)

    def _anneal(self, t):
        if self.vib_schedule == "linear":
            return max(0.0, 1.0 - t / max(1, self.T-1))
        return float(np.exp(-t / max(1, self.T)))

    def run(self):
        X = self.rng.uniform(self.lower, self.upper, (self.N, self.dim))
        f = np.array([self.func(x) for x in X], dtype=float)

        pbest = X.copy()
        pbest_f = f.copy()

        gidx = int(np.argmin(f))
        gbest = X[gidx].copy()
        gbest_f = float(f[gidx])

        c = np.zeros(self.N, dtype=int)

        for t in range(self.T):
            anneal = self._anneal(t)
            pop_std = np.std(X, axis=0)
            pop_std = np.where(pop_std == 0.0, 1e-12, pop_std)

            # robust scale for affinity
            med = np.median(f)
            mad = np.median(np.abs(f - med)) + self.eps
            scale = mad

            for i in range(self.N):
                x = X[i]

                m = 1.0 + np.log1p(c[i])
                if m > self.m_cap:
                    m = self.m_cap

                # HEO update (no r2)
                r1 = self.rng.uniform(1.0 - self.R, 1.0 + self.R, self.dim)
                r3 = self.rng.uniform(0.0, 1.0, self.dim)

                term = x * m * r1
                v_g = (gbest - term) * r3
                v_l = (pbest[i] - term) * (1.0 - r3)
                x_new = x + v_g + v_l

                # annealed vibration
                x_new = x_new + (anneal * self.vib_sigma0) * (pop_std * self.rng.normal(0.0, 1.0, self.dim))

                # ---- Directional CIBA mutation ----
                # bounded affinity in [0,1)
                aff_raw = (f[i] - gbest_f) / (scale + self.eps)
                aff = np.tanh(max(0.0, aff_raw))

                # amplitude: close->0, far->~beta
                delta = self.beta * (1.0 - np.exp(-self.k_aff * aff))

                # direction toward global best + random component
                dir_vec = (gbest - x_new)
                dir_norm = np.linalg.norm(dir_vec) + self.eps
                dir_unit = dir_vec / dir_norm

                rand_vec = self.rng.normal(0.0, 1.0, self.dim)
                mut_vec = self.alpha_dir * dir_unit + (1.0 - self.alpha_dir) * rand_vec

                x_mut = x_new + delta * mut_vec
                x_mut = np.clip(x_mut, self.lower, self.upper)
                f_mut = float(self.func(x_mut))

                if f_mut < f[i]:
                    X[i] = x_mut
                    f[i] = f_mut
                    c[i] = 0
                else:
                    c[i] += 1

                if f[i] < pbest_f[i]:
                    pbest[i] = X[i].copy()
                    pbest_f[i] = f[i]
                if f[i] < gbest_f:
                    gbest = X[i].copy()
                    gbest_f = float(f[i])

                if c[i] > self.c_max:
                    X[i] = self.rng.uniform(-self.bound, self.bound, self.dim)
                    f[i] = float(self.func(X[i]))
                    c[i] = 0

        return gbest, gbest_f

# =========================
# Run protocol
# =========================
DIM=30; BOUND=100.0; SWARM=100; ITERS=1000; RUNS=30

params = dict(
    R=1.0, c_max=30,
    vib_sigma0=0.10, vib_schedule="exp",
    m_cap=10.0,
    beta=5.0, k_aff=2.0, alpha_dir=0.7
)

results=[]
for fname, func in benchmark_funcs:
    vals=[]; times=[]
    print("Running HEO+DirectionalCIBA on", fname, "...")
    for r in range(RUNS):
        opt = HEO_DirectionalCIBA(func, DIM, bound=BOUND, swarm_size=SWARM, max_iters=ITERS, seed=1234+r, **params)
        t0=time.time()
        _, best_f = opt.run()
        t1=time.time()
        vals.append(best_f); times.append(t1-t0)
    vals=np.array(vals); times=np.array(times)
    results.append({
        "Function": fname,
        "HEO+DirCIBA_mean_cost": float(vals.mean()),
        "HEO+DirCIBA_std_cost": float(vals.std()),
        "time_s_per_1000iters": float(times.mean())
    })

df=pd.DataFrame(results)
display(df)

print("\nQuick tuning:")
print("- If Sphere/Step still high: lower beta (2..3) and increase alpha_dir (0.8..0.9).")
print("- If exploration too weak: raise beta (6..10) or lower alpha_dir (0.5..0.7).")
print("- If Schwefel 2.22 spikes: lower m_cap (6..8) and beta (2..4).")


Running HEO+DirectionalCIBA on Sphere ...
Running HEO+DirectionalCIBA on Step ...
Running HEO+DirectionalCIBA on Schwefel 2.21 ...
Running HEO+DirectionalCIBA on Schwefel 2.22 ...
Running HEO+DirectionalCIBA on Rosenbrock ...
Running HEO+DirectionalCIBA on BentCigar ...
Running HEO+DirectionalCIBA on Sumsquares2 ...
Running HEO+DirectionalCIBA on Alpine ...
Running HEO+DirectionalCIBA on Griewank ...
Running HEO+DirectionalCIBA on Rastrigin ...
Running HEO+DirectionalCIBA on Ackley ...
Running HEO+DirectionalCIBA on Levy ...
Running HEO+DirectionalCIBA on Salomon ...
Running HEO+DirectionalCIBA on Schaffer ...


,Function,HEO+DirCIBA_mean_cost,HEO+DirCIBA_std_cost,time_s_per_1000iters
0,Sphere,1.699977e+01,5.693515e+00,6.310749
1,Step,1.903333e+01,4.874993e+00,6.508145
2,Schwefel 2.21,2.383925e+00,3.145120e-01,6.100682
3,Schwefel 2.22,1.894210e+01,1.977327e+00,6.571396
4,Rosenbrock,3.604600e+03,1.746432e+03,7.112556
5,BentCigar,1.680340e+07,5.625960e+06,6.386576
6,Sumsquares2,2.302411e+02,7.196743e+01,6.746630
7,Alpine,1.750522e+01,2.469166e+00,6.658972
8,Griewank,7.556969e-01,1.074187e-01,7.582045
9,Rastrigin,2.630953e+02,1.942363e+01,6.844864



Quick tuning:
- If Sphere/Step still high: lower beta (2..3) and increase alpha_dir (0.8..0.9).
- If exploration too weak: raise beta (6..10) or lower alpha_dir (0.5..0.7).
- If Schwefel 2.22 spikes: lower m_cap (6..8) and beta (2..4).


In [ ]:
import numpy as np
import pandas as pd
import time

# ============================================================
# 1) Base benchmark functions
# ============================================================
def sphere(x): return np.sum(x**2)
def rastrigin(x): return 10.0*x.size + np.sum(x**2 - 10.0*np.cos(2*np.pi*x))
def rosenbrock(x): return np.sum(100.0*(x[1:] - x[:-1]**2)**2 + (1-x[:-1])**2)
def griewank(x):
    i = np.arange(1, x.size+1)
    return np.sum(x**2)/4000.0 - np.prod(np.cos(x/np.sqrt(i))) + 1.0
def levy(x):
    w = 1 + (x - 1)/4
    term1 = np.sin(np.pi*w[0])**2
    term3 = (w[-1]-1)**2 * (1 + np.sin(2*np.pi*w[-1])**2)
    term2 = np.sum((w[:-1]-1)**2 * (1 + 10*np.sin(np.pi*w[:-1] + 1)**2))
    return term1 + term2 + term3

# ============================================================
# 2) Shift + Offset wrapper (non-zero minimum, non-zero optimum)
# f_shifted(x) = f(x - shift) + offset
# Global minimum value = offset at x = shift (or shifted optimum)
# ============================================================
def make_shifted_offset(func, shift, offset):
    shift = np.asarray(shift, dtype=float)
    def wrapped(x):
        return func(np.asarray(x) - shift) + offset
    return wrapped

# ============================================================
# 3) Algorithm A: v2 (HEO+ES 1/5) — bounded escape, no phases, unbiased skip
# ============================================================
class HEO_ES_NoPhases:
    def __init__(
        self,
        func, dim, bound=100.0, swarm_size=100, max_iters=1000,
        c_max=30, R=1.0,
        vib_sigma0=0.10, vib_schedule="exp",
        es_init_sigma_scale=0.10,  # sigma0 = scale*(range)/sqrt(dim)
        es_window=20, es_c=0.85,
        m_cap=10.0,
        seed=None
    ):
        self.func = func
        self.dim = int(dim)
        self.bound = float(bound)
        self.N = int(swarm_size)
        self.T = int(max_iters)
        self.c_max = int(c_max)
        self.R = float(R)

        self.vib_sigma0 = float(vib_sigma0)
        self.vib_schedule = vib_schedule

        self.es_window = int(es_window)
        self.es_c = float(es_c)
        self.es_init_sigma_scale = float(es_init_sigma_scale)

        self.m_cap = float(m_cap)

        self.lower = -self.bound*np.ones(self.dim)
        self.upper =  self.bound*np.ones(self.dim)
        self.range_width = (self.upper - self.lower)[0]

        self.rng = np.random.default_rng(seed)

    def _anneal(self, t):
        if self.vib_schedule == "linear":
            return max(0.0, 1.0 - (t / max(1, self.T - 1)))
        return float(np.exp(-t / max(1, self.T)))

    def run(self, verbose=False, log_every=100):
        X = self.rng.uniform(self.lower, self.upper, size=(self.N, self.dim))
        f = np.array([self.func(x) for x in X], dtype=float)

        pbest = X.copy()
        pbest_f = f.copy()

        gidx = int(np.argmin(f))
        gbest_f = float(f[gidx])

        c = np.zeros(self.N, dtype=int)

        # ES sigma dimension-aware
        sigma0 = self.es_init_sigma_scale * (self.range_width / np.sqrt(self.dim))
        sigma = np.full(self.N, sigma0, dtype=float)
        succ = np.zeros(self.N, dtype=int)
        trials = np.zeros(self.N, dtype=int)

        for t in range(self.T):
            anneal = self._anneal(t)
            pop_std = np.std(X, axis=0)
            pop_std = np.where(pop_std == 0.0, 1e-12, pop_std)

            for i in range(self.N):
                x = X[i]

                m = 1.0 + np.log1p(c[i])
                if m > self.m_cap:
                    m = self.m_cap

                r1 = self.rng.uniform(1.0 - self.R, 1.0 + self.R, size=self.dim)
                r3 = self.rng.uniform(0.0, 1.0, size=self.dim)

                term = x * m * r1
                v_g  = (X[np.argmin(f)] - term) * r3
                v_l  = (pbest[i] - term) * (1.0 - r3)
                x_new = x + v_g + v_l

                # annealed vibration
                x_new = x_new + (anneal * self.vib_sigma0) * (pop_std * self.rng.normal(0.0, 1.0, size=self.dim))
                x_new = np.clip(x_new, self.lower, self.upper)

                # (1+1)-ES
                x_mut = x_new + sigma[i] * self.rng.normal(0.0, 1.0, size=self.dim)
                x_mut = np.clip(x_mut, self.lower, self.upper)
                f_mut = float(self.func(x_mut))

                trials[i] += 1
                if f_mut < f[i]:
                    X[i] = x_mut
                    f[i] = f_mut
                    succ[i] += 1
                    c[i] = 0
                else:
                    c[i] += 1

                # update pbest/gbest
                if f[i] < pbest_f[i]:
                    pbest[i] = X[i].copy()
                    pbest_f[i] = f[i]
                if f[i] < gbest_f:
                    gbest_f = float(f[i])

                # 1/5 success rule
                if trials[i] >= self.es_window:
                    rate = succ[i] / trials[i]
                    if rate > 0.2:
                        sigma[i] /= self.es_c
                    else:
                        sigma[i] *= self.es_c
                    sigma[i] = float(np.clip(sigma[i], 1e-14, self.bound))
                    succ[i] = 0
                    trials[i] = 0

                # unbiased random skip
                if c[i] > self.c_max:
                    X[i] = self.rng.uniform(-self.bound, self.bound, size=self.dim)
                    f[i] = float(self.func(X[i]))
                    c[i] = 0

            if verbose and (t % max(1, log_every) == 0 or t == self.T - 1):
                # recompute global best for display
                gbest_f = float(np.min(f))
                print(f"  iter {t:4d}/{self.T}  best={gbest_f:.6e}")

        # final best
        gbest_f = float(np.min(f))
        return gbest_f

# ============================================================
# 4) Algorithm B: DirCIBA (your best current)
# ============================================================
class HEO_DirectionalCIBA:
    def __init__(
        self,
        func, dim, bound=100.0, swarm_size=100, max_iters=1000,
        c_max=30, R=1.0,
        vib_sigma0=0.10, vib_schedule="exp",
        m_cap=10.0,
        beta=5.0, k_aff=2.0, alpha_dir=0.7,
        eps=1e-12,
        seed=None
    ):
        self.func = func
        self.dim = int(dim)
        self.bound = float(bound)
        self.N = int(swarm_size)
        self.T = int(max_iters)
        self.c_max = int(c_max)
        self.R = float(R)

        self.vib_sigma0 = float(vib_sigma0)
        self.vib_schedule = vib_schedule
        self.m_cap = float(m_cap)

        self.beta = float(beta)
        self.k_aff = float(k_aff)
        self.alpha_dir = float(alpha_dir)
        self.eps = float(eps)

        self.lower = -self.bound*np.ones(self.dim)
        self.upper =  self.bound*np.ones(self.dim)
        self.rng = np.random.default_rng(seed)

    def _anneal(self, t):
        if self.vib_schedule == "linear":
            return max(0.0, 1.0 - t / max(1, self.T-1))
        return float(np.exp(-t / max(1, self.T)))

    def run(self, verbose=False, log_every=100):
        X = self.rng.uniform(self.lower, self.upper, (self.N, self.dim))
        f = np.array([self.func(x) for x in X], dtype=float)

        pbest = X.copy()
        pbest_f = f.copy()

        gbest_idx = int(np.argmin(f))
        gbest = X[gbest_idx].copy()
        gbest_f = float(f[gbest_idx])

        c = np.zeros(self.N, dtype=int)

        for t in range(self.T):
            anneal = self._anneal(t)
            pop_std = np.std(X, axis=0)
            pop_std = np.where(pop_std == 0.0, 1e-12, pop_std)

            # robust scale for affinity
            med = np.median(f)
            mad = np.median(np.abs(f - med)) + self.eps
            scale = mad

            for i in range(self.N):
                x = X[i]

                m = 1.0 + np.log1p(c[i])
                if m > self.m_cap:
                    m = self.m_cap

                r1 = self.rng.uniform(1.0 - self.R, 1.0 + self.R, self.dim)
                r3 = self.rng.uniform(0.0, 1.0, self.dim)

                term = x * m * r1
                v_g = (gbest - term) * r3
                v_l = (pbest[i] - term) * (1.0 - r3)
                x_new = x + v_g + v_l

                x_new = x_new + (anneal * self.vib_sigma0) * (pop_std * self.rng.normal(0.0, 1.0, self.dim))

                # affinity -> mutation amplitude
                aff_raw = (f[i] - gbest_f) / (scale + self.eps)
                aff = np.tanh(max(0.0, aff_raw))
                delta = self.beta * (1.0 - np.exp(-self.k_aff * aff))

                # directional + random mutation
                dir_vec = (gbest - x_new)
                dir_unit = dir_vec / (np.linalg.norm(dir_vec) + self.eps)
                rand_vec = self.rng.normal(0.0, 1.0, self.dim)
                mut_vec = self.alpha_dir * dir_unit + (1.0 - self.alpha_dir) * rand_vec

                x_mut = x_new + delta * mut_vec
                x_mut = np.clip(x_mut, self.lower, self.upper)
                f_mut = float(self.func(x_mut))

                if f_mut < f[i]:
                    X[i] = x_mut
                    f[i] = f_mut
                    c[i] = 0
                else:
                    c[i] += 1

                if f[i] < pbest_f[i]:
                    pbest[i] = X[i].copy()
                    pbest_f[i] = f[i]

                if f[i] < gbest_f:
                    gbest_f = float(f[i])
                    gbest = X[i].copy()

                if c[i] > self.c_max:
                    X[i] = self.rng.uniform(-self.bound, self.bound, self.dim)
                    f[i] = float(self.func(X[i]))
                    c[i] = 0

            if verbose and (t % max(1, log_every) == 0 or t == self.T - 1):
                gbest_f = float(np.min(f))
                print(f"  iter {t:4d}/{self.T}  best={gbest_f:.6e}")

        gbest_f = float(np.min(f))
        return gbest_f

# ============================================================
# 5) Non-zero-minimum suite (fixed shifts so both algos face same tasks)
# ============================================================
DIM = 30
BOUND = 100.0
SWARM_SIZE = 100
MAX_ITERS = 1000
RUNS = 30

# Logging settings
LOG_FIRST_RUN_ONLY = True     # log only run0 for each (algo, function)
LOG_EVERY = max(1, MAX_ITERS // 10)

rng = np.random.default_rng(2025)
SHIFT_SCALE = 30.0
OFFSET = 123.456

shift_vectors = {
    "ShiftedSphere": rng.uniform(-SHIFT_SCALE, SHIFT_SCALE, size=DIM),
    "ShiftedRastrigin": rng.uniform(-SHIFT_SCALE, SHIFT_SCALE, size=DIM),
    "ShiftedRosenbrock": rng.uniform(-2.0, 2.0, size=DIM),
    "ShiftedGriewank": rng.uniform(-SHIFT_SCALE, SHIFT_SCALE, size=DIM),
    "ShiftedLevy": rng.uniform(-SHIFT_SCALE, SHIFT_SCALE, size=DIM),
}

nonzero_suite = [
    ("ShiftedSphere+Offset", make_shifted_offset(sphere, shift_vectors["ShiftedSphere"], OFFSET)),
    ("ShiftedRastrigin+Offset", make_shifted_offset(rastrigin, shift_vectors["ShiftedRastrigin"], OFFSET)),
    ("ShiftedRosenbrock+Offset", make_shifted_offset(rosenbrock, shift_vectors["ShiftedRosenbrock"], OFFSET)),
    ("ShiftedGriewank+Offset", make_shifted_offset(griewank, shift_vectors["ShiftedGriewank"], OFFSET)),
    ("ShiftedLevy+Offset", make_shifted_offset(levy, shift_vectors["ShiftedLevy"], OFFSET)),
]

# ============================================================
# 6) Benchmark runner for both algorithms + logs
# ============================================================
def benchmark(AlgoClass, algo_name, funcs):
    rows = []
    for fname, func in funcs:
        vals = []
        times = []

        print(f"\n=== {algo_name} on {fname} (known min = {OFFSET}) ===")
        for r in range(RUNS):
            verbose = (r == 0) if LOG_FIRST_RUN_ONLY else True

            opt = AlgoClass(
                func=func, dim=DIM, bound=BOUND, swarm_size=SWARM_SIZE, max_iters=MAX_ITERS,
                seed=1234 + r
            )

            t0 = time.time()
            best_f = opt.run(verbose=verbose, log_every=LOG_EVERY if verbose else MAX_ITERS+1)
            t1 = time.time()

            if r == 0:
                gap = best_f - OFFSET
                print(f"  run0 final best = {best_f:.6e}   gap_to_opt = {gap:.6e}")

            vals.append(best_f)
            times.append(t1 - t0)

        vals = np.array(vals, dtype=float)
        times = np.array(times, dtype=float)

        rows.append({
            "Algo": algo_name,
            "Function": fname,
            "mean_best": float(vals.mean()),
            "std_best": float(vals.std()),
            "time_s_per_1000iters": float(times.mean()),
            "known_global_min": OFFSET,
            "mean_gap_to_opt": float(vals.mean() - OFFSET),
        })
    return rows

rows = []
rows += benchmark(HEO_ES_NoPhases, "v2_HEO+ES(1/5)", nonzero_suite)
rows += benchmark(HEO_DirectionalCIBA, "DirCIBA", nonzero_suite)

df = pd.DataFrame(rows)
print("\n=== Full Results ===")
display(df)

pivot_mean = df.pivot(index="Function", columns="Algo", values="mean_best")
pivot_gap  = df.pivot(index="Function", columns="Algo", values="mean_gap_to_opt")

print("\n=== Mean Best (side-by-side) ===")
display(pivot_mean)

print("\n=== Mean Gap to Optimum (mean_best - known_global_min) ===")
display(pivot_gap)

print("\nInterpretation:")
print(f"- True global minimum is {OFFSET} for ALL functions above.")
print("- Smaller gap is better; gap≈0 means the algorithm truly found the optimum region.")



=== v2_HEO+ES(1/5) on ShiftedSphere+Offset (known min = 123.456) ===
  iter    0/1000  best=3.624814e+04
  iter  100/1000  best=3.958654e+03
  iter  200/1000  best=3.136181e+03
  iter  300/1000  best=3.042455e+03
  iter  400/1000  best=3.797053e+03
  iter  500/1000  best=2.987356e+03
  iter  600/1000  best=3.944240e+03
  iter  700/1000  best=3.191132e+03
  iter  800/1000  best=3.276589e+03
  iter  900/1000  best=3.299118e+03
  iter  999/1000  best=2.571555e+03
  run0 final best = 2.571555e+03   gap_to_opt = 2.448099e+03

=== v2_HEO+ES(1/5) on ShiftedRastrigin+Offset (known min = 123.456) ===
  iter    0/1000  best=2.410514e+04
  iter  100/1000  best=4.166052e+03
  iter  200/1000  best=3.035721e+03
  iter  300/1000  best=3.144204e+03
  iter  400/1000  best=2.407935e+03
  iter  500/1000  best=2.661381e+03
  iter  600/1000  best=3.051842e+03
  iter  700/1000  best=2.660196e+03
  iter  800/1000  best=3.039007e+03
  iter  900/1000  best=2.738781e+03
  iter  999/1000  best=2.228485e+03
  ru

,Algo,Function,mean_best,std_best,time_s_per_1000iters,known_global_min,mean_gap_to_opt
0,v2_HEO+ES(1/5),ShiftedSphere+Offset,2735.099031,338.926159,5.694152,123.456,2611.643031
1,v2_HEO+ES(1/5),ShiftedRastrigin+Offset,2502.983907,285.393404,5.891687,123.456,2379.527907
2,v2_HEO+ES(1/5),ShiftedRosenbrock+Offset,25567.769690,11082.744258,6.141751,123.456,25444.313690
3,v2_HEO+ES(1/5),ShiftedGriewank+Offset,124.851751,0.060998,6.612759,123.456,1.395751
4,v2_HEO+ES(1/5),ShiftedLevy+Offset,745.447491,129.503290,8.003341,123.456,621.991491
5,DirCIBA,ShiftedSphere+Offset,2232.965160,325.901819,6.420023,123.456,2109.509160
6,DirCIBA,ShiftedRastrigin+Offset,2207.814144,259.809843,7.008132,123.456,2084.358144
7,DirCIBA,ShiftedRosenbrock+Offset,15334.478615,4115.509250,7.244304,123.456,15211.022615
8,DirCIBA,ShiftedGriewank+Offset,124.832337,0.046865,7.711202,123.456,1.376337
9,DirCIBA,ShiftedLevy+Offset,896.434436,202.447140,9.045990,123.456,772.978436



=== Mean Best (side-by-side) ===


Algo,DirCIBA,v2_HEO+ES(1/5)
Function,,
ShiftedGriewank+Offset,124.832337,124.851751
ShiftedLevy+Offset,896.434436,745.447491
ShiftedRastrigin+Offset,2207.814144,2502.983907
ShiftedRosenbrock+Offset,15334.478615,25567.769690
ShiftedSphere+Offset,2232.965160,2735.099031



=== Mean Gap to Optimum (mean_best - known_global_min) ===


Algo,DirCIBA,v2_HEO+ES(1/5)
Function,,
ShiftedGriewank+Offset,1.376337,1.395751
ShiftedLevy+Offset,772.978436,621.991491
ShiftedRastrigin+Offset,2084.358144,2379.527907
ShiftedRosenbrock+Offset,15211.022615,25444.313690
ShiftedSphere+Offset,2109.509160,2611.643031



Interpretation:
- True global minimum is 123.456 for ALL functions above.
- Smaller gap is better; gap≈0 means the algorithm truly found the optimum region.


In [1]:
import numpy as np
import pandas as pd
import time

# ============================================================
# 1) Base benchmark functions
# ============================================================
def sphere(x): return np.sum(x**2)
def rastrigin(x): return 10.0*x.size + np.sum(x**2 - 10.0*np.cos(2*np.pi*x))
def rosenbrock(x): return np.sum(100.0*(x[1:] - x[:-1]**2)**2 + (1-x[:-1])**2)
def griewank(x):
    i = np.arange(1, x.size+1)
    return np.sum(x**2)/4000.0 - np.prod(np.cos(x/np.sqrt(i))) + 1.0
def levy(x):
    w = 1 + (x - 1)/4
    term1 = np.sin(np.pi*w[0])**2
    term3 = (w[-1]-1)**2 * (1 + np.sin(2*np.pi*w[-1])**2)
    term2 = np.sum((w[:-1]-1)**2 * (1 + 10*np.sin(np.pi*w[:-1] + 1)**2))
    return term1 + term2 + term3

def make_shifted_offset(func, shift, offset):
    shift = np.asarray(shift, dtype=float)
    def wrapped(x):
        return func(np.asarray(x) - shift) + offset
    return wrapped

# ============================================================
# 2) DirCIBA + Differential Escape (Option A)
#    - HEO-style term (bounded) stays
#    - Add DE-style escape direction: gamma*(X[r1]-X[r2])
#    - Directional CIBA mutation stays
#    - Vibration anneals to 0
# ============================================================
class HEO_DirCIBA_DEEscape:
    def __init__(
        self,
        func, dim,
        bound=100.0, swarm_size=100, max_iters=1000,
        c_max=30, R=1.0,
        # vibration
        vib_sigma0=0.10, vib_schedule="exp",
        # bounded escape multiplier
        m_cap=10.0,
        # DirCIBA mutation
        beta=5.0, k_aff=2.0, alpha_dir=0.7,
        # Differential escape kick
        gamma=0.8,             # strength of differential vector
        p_de=0.7,              # probability to apply DE kick per particle update
        eps=1e-12,
        seed=None
    ):
        self.func = func
        self.dim = int(dim)
        self.bound = float(bound)
        self.N = int(swarm_size)
        self.T = int(max_iters)

        self.c_max = int(c_max)
        self.R = float(R)

        self.vib_sigma0 = float(vib_sigma0)
        self.vib_schedule = vib_schedule

        self.m_cap = float(m_cap)

        self.beta = float(beta)
        self.k_aff = float(k_aff)
        self.alpha_dir = float(alpha_dir)

        self.gamma = float(gamma)
        self.p_de = float(p_de)

        self.eps = float(eps)

        self.lower = -self.bound * np.ones(self.dim)
        self.upper =  self.bound * np.ones(self.dim)
        self.rng = np.random.default_rng(seed)

    def _anneal(self, t):
        if self.vib_schedule == "linear":
            return max(0.0, 1.0 - t / max(1, self.T - 1))
        return float(np.exp(-t / max(1, self.T)))

    def run(self, verbose=False, log_every=100):
        X = self.rng.uniform(self.lower, self.upper, (self.N, self.dim))
        f = np.array([self.func(x) for x in X], dtype=float)

        pbest = X.copy()
        pbest_f = f.copy()

        gidx = int(np.argmin(f))
        gbest = X[gidx].copy()
        gbest_f = float(f[gidx])

        c = np.zeros(self.N, dtype=int)

        for t in range(self.T):
            anneal = self._anneal(t)
            pop_std = np.std(X, axis=0)
            pop_std = np.where(pop_std == 0.0, 1e-12, pop_std)

            # robust scale for affinity
            med = np.median(f)
            mad = np.median(np.abs(f - med)) + self.eps
            scale = mad

            for i in range(self.N):
                x = X[i]

                # bounded escape multiplier (still grows with stagnation)
                m = 1.0 + np.log1p(c[i])
                if m > self.m_cap:
                    m = self.m_cap

                r3 = self.rng.uniform(0.0, 1.0, self.dim)

                # HEO-style escape term
                r1 = self.rng.uniform(1.0 - self.R, 1.0 + self.R, self.dim)
                term = x * m * r1

                v_g = (gbest - term) * r3
                v_l = (pbest[i] - term) * (1.0 - r3)
                x_new = x + v_g + v_l

                # Differential escape kick (population geometry)
                if self.rng.random() < self.p_de:
                    r1i, r2i = self.rng.integers(0, self.N, size=2)
                    while r2i == r1i:
                        r2i = int(self.rng.integers(0, self.N))
                    de_vec = X[r1i] - X[r2i]
                    x_new = x_new + self.gamma * de_vec

                # Annealed vibration (vanishes)
                x_new = x_new + (anneal * self.vib_sigma0) * (pop_std * self.rng.normal(0.0, 1.0, self.dim))

                # DirCIBA mutation (directional)
                aff_raw = (f[i] - gbest_f) / (scale + self.eps)
                aff = np.tanh(max(0.0, aff_raw))
                delta = self.beta * (1.0 - np.exp(-self.k_aff * aff))

                dir_vec = (gbest - x_new)
                dir_unit = dir_vec / (np.linalg.norm(dir_vec) + self.eps)
                rand_vec = self.rng.normal(0.0, 1.0, self.dim)
                mut_vec = self.alpha_dir * dir_unit + (1.0 - self.alpha_dir) * rand_vec

                x_mut = x_new + delta * mut_vec
                x_mut = np.clip(x_mut, self.lower, self.upper)
                f_mut = float(self.func(x_mut))

                if f_mut < f[i]:
                    X[i] = x_mut
                    f[i] = f_mut
                    c[i] = 0
                else:
                    c[i] += 1

                # Update personal best
                if f[i] < pbest_f[i]:
                    pbest[i] = X[i].copy()
                    pbest_f[i] = f[i]

                # Update global best
                if f[i] < gbest_f:
                    gbest_f = float(f[i])
                    gbest = X[i].copy()

                # Random skip if too stagnant (escape!)
                if c[i] > self.c_max:
                    X[i] = self.rng.uniform(-self.bound, self.bound, self.dim)
                    f[i] = float(self.func(X[i]))
                    c[i] = 0

            if verbose and (t % max(1, log_every) == 0 or t == self.T - 1):
                print(f"  iter {t:4d}/{self.T}  best={gbest_f:.6e}")

        return gbest_f

# ============================================================
# 3) Build non-zero-minimum suite (fixed shifts)
# ============================================================
DIM = 30
BOUND = 100.0
SWARM_SIZE = 100
MAX_ITERS = 1000
RUNS = 30

LOG_FIRST_RUN_ONLY = True
LOG_EVERY = max(1, MAX_ITERS // 10)

rng = np.random.default_rng(2025)
SHIFT_SCALE = 30.0
OFFSET = 123.456

shift_vectors = {
    "ShiftedSphere": rng.uniform(-SHIFT_SCALE, SHIFT_SCALE, size=DIM),
    "ShiftedRastrigin": rng.uniform(-SHIFT_SCALE, SHIFT_SCALE, size=DIM),
    "ShiftedRosenbrock": rng.uniform(-2.0, 2.0, size=DIM),
    "ShiftedGriewank": rng.uniform(-SHIFT_SCALE, SHIFT_SCALE, size=DIM),
    "ShiftedLevy": rng.uniform(-SHIFT_SCALE, SHIFT_SCALE, size=DIM),
}

nonzero_suite = [
    ("ShiftedSphere+Offset", make_shifted_offset(sphere, shift_vectors["ShiftedSphere"], OFFSET)),
    ("ShiftedRastrigin+Offset", make_shifted_offset(rastrigin, shift_vectors["ShiftedRastrigin"], OFFSET)),
    ("ShiftedRosenbrock+Offset", make_shifted_offset(rosenbrock, shift_vectors["ShiftedRosenbrock"], OFFSET)),
    ("ShiftedGriewank+Offset", make_shifted_offset(griewank, shift_vectors["ShiftedGriewank"], OFFSET)),
    ("ShiftedLevy+Offset", make_shifted_offset(levy, shift_vectors["ShiftedLevy"], OFFSET)),
]

# ============================================================
# 4) Run benchmark with logs
# ============================================================
results = []
for fname, func in nonzero_suite:
    vals=[]
    times=[]
    print(f"\n=== DirCIBA+DEEscape on {fname} (known min={OFFSET}) ===")
    for r in range(RUNS):
        verbose = (r == 0) if LOG_FIRST_RUN_ONLY else False
        opt = HEO_DirCIBA_DEEscape(
            func=func, dim=DIM,
            bound=BOUND, swarm_size=SWARM_SIZE, max_iters=MAX_ITERS,
            seed=1234+r,
            # You can tweak these if desired:
            gamma=0.8, p_de=0.7,   # DE escape kick
            beta=5.0, k_aff=2.0, alpha_dir=0.7,  # DirCIBA mutation
            m_cap=10.0, vib_sigma0=0.10
        )

        t0=time.time()
        best_f = opt.run(verbose=verbose, log_every=LOG_EVERY if verbose else MAX_ITERS+1)
        t1=time.time()

        if r == 0:
            print(f"  run0 final best={best_f:.6e}  gap_to_opt={best_f-OFFSET:.6e}")

        vals.append(best_f)
        times.append(t1-t0)

    vals=np.array(vals, dtype=float)
    times=np.array(times, dtype=float)

    results.append({
        "Algo": "DirCIBA+DEEscape",
        "Function": fname,
        "mean_best": float(vals.mean()),
        "std_best": float(vals.std()),
        "time_s_per_1000iters": float(times.mean()),
        "known_global_min": OFFSET,
        "mean_gap_to_opt": float(vals.mean() - OFFSET)
    })

df = pd.DataFrame(results)
print("\n=== Results ===")
display(df)

print("\nInterpretation:")
print(f"- True global minimum is {OFFSET} for ALL functions above.")
print("- Smaller mean_gap_to_opt means better generalization beyond zero-min functions.")



=== DirCIBA+DEEscape on ShiftedSphere+Offset (known min=123.456) ===
  iter    0/1000  best=4.459669e+04
  iter  100/1000  best=1.595530e+04
  iter  200/1000  best=1.595530e+04
  iter  300/1000  best=1.274541e+04
  iter  400/1000  best=6.806796e+03
  iter  500/1000  best=6.806796e+03
  iter  600/1000  best=6.806796e+03
  iter  700/1000  best=6.806796e+03
  iter  800/1000  best=6.796846e+03
  iter  900/1000  best=6.796846e+03
  iter  999/1000  best=6.796846e+03
  run0 final best=6.796846e+03  gap_to_opt=6.673390e+03

=== DirCIBA+DEEscape on ShiftedRastrigin+Offset (known min=123.456) ===
  iter    0/1000  best=5.052197e+04
  iter  100/1000  best=2.357508e+04
  iter  200/1000  best=1.245516e+04
  iter  300/1000  best=9.541353e+03
  iter  400/1000  best=9.541353e+03
  iter  500/1000  best=8.397918e+03
  iter  600/1000  best=8.397918e+03
  iter  700/1000  best=7.389316e+03
  iter  800/1000  best=7.389316e+03
  iter  900/1000  best=7.389316e+03
  iter  999/1000  best=7.389316e+03
  run0 fi

,Algo,Function,mean_best,std_best,time_s_per_1000iters,known_global_min,mean_gap_to_opt
0,DirCIBA+DEEscape,ShiftedSphere+Offset,6.738098e+03,9.474128e+02,9.566930,123.456,6.614642e+03
1,DirCIBA+DEEscape,ShiftedRastrigin+Offset,5.836202e+03,9.478524e+02,9.505084,123.456,5.712746e+03
2,DirCIBA+DEEscape,ShiftedRosenbrock+Offset,9.840991e+07,8.478183e+07,14.150385,123.456,9.840978e+07
3,DirCIBA+DEEscape,ShiftedGriewank+Offset,1.256748e+02,1.645009e-01,10.150678,123.456,2.218775e+00
4,DirCIBA+DEEscape,ShiftedLevy+Offset,2.746488e+03,6.437072e+02,11.508096,123.456,2.623032e+03



Interpretation:
- True global minimum is 123.456 for ALL functions above.
- Smaller mean_gap_to_opt means better generalization beyond zero-min functions.
